# Show Reel — Community Persona Pipeline (Vertex AI **Batch** + **Multimodal**)

**Platform:** Instagram   **Runtime:** Google Colab Enterprise / Vertex AI (ADC auth)

This is the batch-inference rewrite of the persona pipeline. Both LLM stages now run as
**asynchronous Vertex AI batch jobs** (unified `google-genai` SDK): build a JSONL of requests →
upload to GCS → `client.batches.create` → poll → retrieve the sharded JSONL responses. Batch is
~50% cheaper than online calls and sidesteps per-request rate limits.

Persona identification is **multimodal**: each user request carries the media of the posts they
engage with most, read straight from GCS via `gs://` `fileData` parts —
- **image / carousel** posts → the photo(s) + metadata (tagged users, music, caption);
- **reel / feed / carousel_video** posts → the sampled `frames/*.jpg` + the `transcription.txt` + metadata.

**Flow:** Stage 0 features → Stage 1 taxonomy discovery (Flash, batch) → human approval →
Stage 2 classification (Pro, batch). Set `PIPELINE_MODE` in config.

## Install Dependencies

Run once per runtime, then restart the kernel. The unified `google-genai` SDK drives both the
batch jobs and the online connectivity test; `google-cloud-storage` handles GCS upload/list/download.

In [1]:
!pip install -q google-genai google-cloud-storage gcsfs pandas pyarrow tqdm emoji

## Configuration

Everything environment- and cost-related lives here. The multimodal caps (`MAX_MEDIA_POSTS_PER_USER`,
`MAX_IMAGES_PER_POST`) directly drive token cost — raise them for richer context, lower them to save money.

In [2]:
import os

# ============================ PLATFORM TOGGLE ============================
# Run the whole persona pipeline against a different platform by changing this.
# Comments come from the multi-platform prepared exports and are filtered to PLATFORM.
PLATFORM = "instagram"        # instagram | youtube | tiktok | facebook
# ========================================================================

# GCP / Vertex AI
GCP_PROJECT_ID = "project-a9b99a62-b082-46aa-b54"
GCP_LOCATION   = "europe-west1"
GCS_BUCKET     = "afb_project2"

# --- Prepared comments (ALL platforms; filtered to PLATFORM at load) -----------
# comments_ml      : numeric features + bare-numeric author_id/media_id + comment_id (ig_comment_<n>)
# edges_replies_to : reply structure (src --replies to--> dst), joined on comment_id
# comments_llm     : raw text keyed by comment_id (no author_id -> join via comment_id)
COMMENTS_ML_PATH   = "outputs/comments_ml.parquet"
COMMENTS_LLM_PATH  = "outputs/comments_llm.jsonl"
EDGES_REPLIES_PATH = "outputs/edges_replies_to.parquet"

# --- Multimodal context (Instagram only): post metadata + media on GCS ---------
# ig_multimodal_final bridges media_id <-> shortcode and carries post metadata
# (tagged users, music, caption); media objects live under MEDIA_GCS_PREFIX.
MEDIA_INDEX_PATH = "outputs/ig_multimodal_final.parquet"
MEDIA_GCS_PREFIX = "multimodal_dataset_fixed/"     # gs://bucket/<prefix>/<form>/<shortcode>/...
ATTACH_MEDIA     = False  #(PLATFORM == "instagram")        # media context is wired for IG; others run comments-only

# Models — Flash discovers the taxonomy on a sample, Pro classifies the full set.
MODEL_STAGE1_EXPLORATORY = "gemini-2.5-flash"
MODEL_STAGE2_CLASSIFY    = "gemini-2.5-pro"

# Determinism (academic reproducibility)
TEMPERATURE              = 0.0
TOP_P                    = 1.0
# Gemini 2.5 are THINKING models: thinking tokens count against maxOutputTokens, so these must be
# big enough for (thinking + JSON output) or the response truncates (finishReason=MAX_TOKENS).
MAX_OUTPUT_TOKENS_STAGE1 = 24576   # raised: 8192 truncated multi-persona JSON arrays mid-string
MAX_OUTPUT_TOKENS_STAGE2 = 2048
THINKING_BUDGET_STAGE1   = 1024   # Flash: cap thinking so the JSON array always completes (0 = disable)
THINKING_BUDGET_STAGE2   = 512    # Pro:   min 128; keep modest for cost across many requests

# Pipeline mode:  SAMPLE = Stage 0+1 (discover, then pause for approval)
#                 FULL   = Stage 0+2 (classify, needs APPROVED taxonomy)
#                 ALL    = both in sequence
PIPELINE_MODE = "SAMPLE"   # SAMPLE | FULL | ALL

# Sampling / batching
SAMPLE_N_USERS           = 20000
SAMPLE_SEED              = 42
STAGE1_USERS_PER_REQUEST = 3
MAX_BATCH_TOKENS         = 900_000   # stay under Vertex 1M token limit; each job gets split
STAGE2_MAX_USERS         = None

# Multimodal attachment caps (cost control)
MAX_MEDIA_POSTS_PER_USER = 2
MAX_IMAGES_PER_POST      = 2
INCLUDE_TRANSCRIPT       = True
MAX_TRANSCRIPT_CHARS     = 1500

# Batch I/O on GCS
BATCH_INPUT_PREFIX    = "persona_batch/input/"
BATCH_OUTPUT_PREFIX   = "persona_batch/output/"
POLL_INTERVAL_SECONDS = 60

# Local artifacts (uploaded to GCS after the run)
LOCAL_DIR          = "outputs"
TAXONOMY_JSON_PATH = f"{LOCAL_DIR}/taxonomy.json"
RESULTS_PATH       = f"{LOCAL_DIR}/user_personas.parquet"
os.makedirs(LOCAL_DIR, exist_ok=True)


# --- Sentiment pipeline outputs (consumed by Stage 0 room-vibe enrichment) ----
# sentiment_<platform>.parquet is written by sentiment_pipeline.ipynb retrieve step.
# post_vibes.parquet is a post-level summary derived from it (one row per media_id).
SENTIMENT_PATH  = f"{LOCAL_DIR}/sentiment_{PLATFORM}.parquet"
POST_VIBES_PATH = f"{LOCAL_DIR}/post_vibes_{PLATFORM}.parquet"

print("✅ Configuration loaded.")
print(f"   Platform: {PLATFORM}  (media context: {'ON' if ATTACH_MEDIA else 'OFF'})")
print(f"   Bucket:   gs://{GCS_BUCKET}")
print(f"   Mode:     {PIPELINE_MODE}")

✅ Configuration loaded.
   Platform: instagram  (media context: OFF)
   Bucket:   gs://afb_project2
   Mode:     SAMPLE


## Vertex AI Client (google-genai)

`genai.Client(vertexai=True, ...)` flips the unified SDK to the Vertex backend (IAM/ADC auth, GCS
I/O, regional endpoints) — the same client submits batch jobs and runs the online connectivity test.

In [3]:
from google import genai
from google.genai.types import CreateBatchJobConfig, JobState

client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)

def gen_config_dict(max_tokens: int, thinking_budget: int | None = None) -> dict:
    """Per-request generationConfig (camelCase REST keys) embedded in each JSONL line.
    responseMimeType forces JSON; thinkingConfig caps the 2.5 models' thinking-token spend so
    the visible output isn't starved (the MAX_TOKENS truncation bug)."""
    cfg = {
        "temperature": TEMPERATURE,
        "topP": TOP_P,
        "maxOutputTokens": max_tokens,
        "responseMimeType": "application/json",
    }
    if thinking_budget is not None:
        cfg["thinkingConfig"] = {"thinkingBudget": thinking_budget}
    return cfg

print("✅ genai Vertex client ready:", GCP_PROJECT_ID, GCP_LOCATION)

✅ genai Vertex client ready: project-a9b99a62-b082-46aa-b54 europe-west1


## Data Loading

Persona features come from the **comments dataset** — the multi-platform prepared exports, filtered
to `PLATFORM`:
- `Preped_Comments/comments_ml.parquet` — per-comment numeric features + bare-numeric `author_id`/`media_id`
- `HeteroGraph/edges_replies_to.parquet` — reply structure (`src --replies to--> dst`), joined on `comment_id`

For Instagram, `ig_multimodal_final.parquet` (post metadata + `media_id`↔`shortcode`) and the GCS media
are loaded as **multimodal context** only. Other platforms run comments-only.

In [4]:
import pandas as pd

print(f"Loading prepared comments for platform = {PLATFORM!r}")

# 1) Comment feature matrix — predicate-pushdown filter to PLATFORM (skips other platforms).
ig_comments = pd.read_parquet(COMMENTS_ML_PATH, filters=[("platform", "==", PLATFORM)])
ig_comments["author_id"] = ig_comments["author_id"].astype(str)
ig_comments["media_id"]  = ig_comments["media_id"].astype(str)
print(f"  comments_ml[{PLATFORM}]: {len(ig_comments):,} comments | "
      f"{ig_comments['author_id'].nunique():,} authors | {ig_comments['media_id'].nunique():,} media")

# 2) Reply structure: edges_replies_to (src --replies to--> dst), joined on comment_id.
try:
    replies = pd.read_parquet(EDGES_REPLIES_PATH, filters=[("platform", "==", PLATFORM)],
                              columns=["src_comment_id", "dst_comment_id"])
    replies = (replies.rename(columns={"src_comment_id": "comment_id",
                                       "dst_comment_id": "reply_to_comment_id"})
                      .drop_duplicates("comment_id"))
    ig_comments = ig_comments.merge(replies, on="comment_id", how="left")
    print(f"  reply edges: {len(replies):,} | replies flagged on "
          f"{ig_comments['reply_to_comment_id'].notna().sum():,} comments")
except Exception as e:
    print("  ⚠️  edges_replies_to unavailable -> no reply features:", str(e)[:120])
    ig_comments["reply_to_comment_id"] = pd.NA

# 3) Instagram-only multimodal context: post metadata + media index.
if ATTACH_MEDIA:
    media_index = pd.read_parquet(MEDIA_INDEX_PATH)
    media_index = media_index[media_index["media_id"].notna()].copy()
    media_index["media_id"] = media_index["media_id"].astype(str)
    ig_media = media_index                      # post-metadata source for Stage 0 temporal features
    print(f"  media_index: {len(media_index):,} posts | {media_index['shortcode'].nunique():,} shortcodes")
else:
    media_index = None
    ig_media = None
    print(f"  media context OFF for platform={PLATFORM} -> comments-only run")

# 4) Validation
n_before = len(ig_comments)
ig_comments = ig_comments[ig_comments["author_id"].notna() & (ig_comments["author_id"] != "")].copy()
print(f"\n✅ Comments ready: {len(ig_comments):,}/{n_before:,} | "
      f"unique authors: {ig_comments['author_id'].nunique():,}")

Loading prepared comments for platform = 'instagram'
  comments_ml[instagram]: 499,752 comments | 194,513 authors | 1,501 media
  ⚠️  edges_replies_to unavailable -> no reply features: [Errno 2] No such file or directory: 'outputs/edges_replies_to.parquet'
  media context OFF for platform=instagram -> comments-only run

✅ Comments ready: 499,752/499,752 | unique authors: 194,513


## Stage 0 — Feature Engineering

All comment-level features (word count, emoji count, etc.) are pre-computed upstream. Here we derive
binary flags + temporal features, aggregate to one row per `author_id`, and attach each user's
representative comment text.

In [5]:
# ── canonical 3-class sentiment ───────────────────────────────────────────────
# The sentiment LLM occasionally returns free-form emotion labels instead of the
# 3-class schema. canon_sentiment() collapses any Series to positive/neutral/negative
# so stratification, the user-sentiment cache, and the plots all agree.
SENT_CANON_MAP = {
    "positive": "positive", "negative": "negative", "neutral": "neutral",
    "amused": "positive", "amusement": "positive", "joke": "positive",
    "joy": "positive", "support": "positive", "supportive": "positive",
    "affection": "positive", "anticipation": "positive",
    "sarcastic": "negative", "sarcasm": "negative", "sadness": "negative",
    "question": "neutral", "suggestion": "neutral", "surprise": "neutral",
    "mixed": "neutral", "spam": "neutral",
}
def canon_sentiment(series):
    s = series.astype(str).str.strip().str.lower()
    return s.map(SENT_CANON_MAP).fillna("neutral")

import numpy as np
from datetime import timedelta
import os

# === FEATURE ENGINEERING (always runs) ===
# 1. Binary flags from pre-computed counts
ig_comments["has_emoji"]    = (ig_comments["emoji_count"] > 0).astype(int)
ig_comments["has_question"] = (ig_comments["question_count"] > 0).astype(int)
ig_comments["has_exclaim"]  = (ig_comments["exclamation_count"] > 0).astype(int)
ig_comments["is_reply"]     = ig_comments["reply_to_comment_id"].notna().astype(int)
print("Binary flags derived.")

# Join per-comment sentiment/toxicity from sentiment pipeline output.
# sentiment_score: -1 (very negative) -> +1 (very positive)
# is_toxic: 1 if toxicity in {mild, severe}
# is_negative: 1 if sentiment == negative
_sent_path = SENTIMENT_PATH
if os.path.exists(_sent_path):
    _sent = pd.read_parquet(
        _sent_path,
        columns=["comment_id", "sentiment_score", "toxicity", "sentiment"],
    )
    # collapse free-form emotion labels to canonical 3-class (see canon_sentiment)
    _sent["sentiment"] = canon_sentiment(_sent["sentiment"])
    _sent["is_toxic"]    = _sent["toxicity"].isin(["mild", "severe"]).astype(int)
    _sent["is_negative"] = (_sent["sentiment"] == "negative").astype(int)
    ig_comments = ig_comments.merge(
        _sent[["comment_id", "sentiment_score", "is_toxic", "is_negative"]],
        on="comment_id", how="left",
    )
    ig_comments["sentiment_score"] = ig_comments["sentiment_score"].fillna(0.0)
    ig_comments["is_toxic"]        = ig_comments["is_toxic"].fillna(0).astype(int)
    ig_comments["is_negative"]     = ig_comments["is_negative"].fillna(0).astype(int)
    _n_sent = _sent["comment_id"].nunique()
    print(f"Sentiment joined: {_n_sent:,} comments enriched.")
else:
    ig_comments["sentiment_score"] = 0.0
    ig_comments["is_toxic"]        = 0
    ig_comments["is_negative"]     = 0
    print(f"WARNING: {_sent_path} not found — sentiment features will be zero.")
# Guard: ensure columns exist even if the branch above was skipped for any reason.
for _col, _default in [("sentiment_score", 0.0), ("is_toxic", 0), ("is_negative", 0)]:
    if _col not in ig_comments.columns:
        ig_comments[_col] = _default

# 2. Temporal features vs. post time (uses Instagram post metadata when available)
ig_comments["timestamp"] = pd.to_datetime(ig_comments["timestamp"], errors="coerce", utc=True)
if ATTACH_MEDIA and ig_media is not None and "timestamp" in ig_media.columns:
    pm = ig_media[["media_id", "timestamp"]].rename(columns={"timestamp": "post_timestamp"})
    ig_comments = ig_comments.drop(columns=[c for c in ["post_timestamp","hours_to_comment"] if c in ig_comments.columns])
    ig_comments = ig_comments.merge(pm, on="media_id", how="left")
    ig_comments["post_timestamp"]  = pd.to_datetime(ig_comments["post_timestamp"], errors="coerce", utc=True)
    ig_comments["hours_to_comment"] = (
        (ig_comments["timestamp"] - ig_comments["post_timestamp"]).dt.total_seconds() / 3600
    ).clip(lower=0)
else:
    ig_comments["hours_to_comment"] = np.nan
    print("   (no post metadata -> hours_to_comment = NaN)")


# === CACHED AGGREGATION (user_features computation) ===
USER_FEATURES_CACHE_PATH = f"{LOCAL_DIR}/user_features_{PLATFORM}_cache.parquet"

if os.path.exists(USER_FEATURES_CACHE_PATH):
    print(f"Loading cached user_features from {USER_FEATURES_CACHE_PATH} ...")
    user_features = pd.read_parquet(USER_FEATURES_CACHE_PATH)
    print(f"✓ Cached user_features loaded: {len(user_features):,} users")
else:
    print("Computing user_features (no cache found) ...")
    
    # 3. Room-vibe enrichment — load post-level sentiment summary from sentiment pipeline output
    import logging
    
    _log = logging.getLogger("persona_pipeline")
    
    def load_post_vibes(path: str) -> "pd.DataFrame | None":
        """
        Load post-level room-vibe metrics written by sentiment_pipeline.ipynb.
        Expected columns: media_id, room_vibe, room_consensus, stance_alignment.
        Returns None (and logs a warning) if the file is absent — pipeline continues
        without room-vibe features so it does not block pre-sentiment runs.
        """
        if not os.path.exists(path):
            _log.warning(
                "POST_VIBES_PATH not found (%s) — room-vibe features will be absent. "
                "Run sentiment_pipeline retrieve step first.", path
            )
            return None
        df = pd.read_parquet(path, columns=["media_id", "room_vibe", "room_consensus", "stance_alignment"])
        df["media_id"] = df["media_id"].astype(str)
        _log.info("post_vibes loaded: %d posts", len(df))
        return df
    
    _post_vibes = load_post_vibes(POST_VIBES_PATH)
    
    
    # 4. User-level aggregation
    def build_user_feature_matrix(df: pd.DataFrame, post_vibes: "pd.DataFrame | None" = None) -> pd.DataFrame:
        grp = df.groupby("author_id")
        agg = {
            "total_comments":          grp.size(),
            "unique_posts_commented":  grp["media_id"].nunique(),
            "total_replies_made":      grp["is_reply"].sum(),
            "reply_ratio":             grp["is_reply"].mean(),
            "mean_hours_to_comment":   grp["hours_to_comment"].mean(),
            "median_hours_to_comment": grp["hours_to_comment"].median(),
            "pct_comments_under_1h":   grp["hours_to_comment"].apply(lambda x: (x < 1).mean()),
            "pct_comments_under_24h":  grp["hours_to_comment"].apply(lambda x: (x < 24).mean()),
            "activity_span_days":      grp["timestamp"].apply(lambda x: (x.max() - x.min()).days if x.notna().any() else 0),
            "mean_word_count":         grp["word_count"].mean(),
            "mean_mention_count":      grp["mention_count"].mean(),
            "emoji_usage_rate":        grp["has_emoji"].mean(),
            "question_rate":           grp["has_question"].mean(),
            "exclamation_rate":        grp["has_exclaim"].mean(),
            "mean_sentiment_score":    grp["sentiment_score"].mean(),
            "pct_negative":            grp["is_negative"].mean(),
            "toxicity_rate":           grp["is_toxic"].mean(),
        }
        feat = pd.DataFrame(agg).reset_index()
        feat["post_concentration_ratio"] = (
            feat["unique_posts_commented"] / feat["total_comments"]
        ).clip(upper=1.0)
        # Fill timing features that are NaN when post metadata is absent (non-IG platforms).
        for c in ["mean_hours_to_comment", "median_hours_to_comment",
                  "pct_comments_under_1h", "pct_comments_under_24h"]:
            feat[c] = feat[c].fillna(0)

        # ── Room-vibe enrichment (IG only; skipped gracefully when post_vibes is None) ──
        if post_vibes is not None:
            # Join comments to per-post vibe metrics, then aggregate per user.
            enriched = df[["author_id", "media_id"]].merge(
                post_vibes, on="media_id", how="left"
            )
            vibe_grp = enriched.groupby("author_id")

            # mean consensus and sponsorship tolerance across posts the user engaged with
            feat = feat.merge(
                vibe_grp["room_consensus"].mean().rename("mean_engaged_consensus"),
                on="author_id", how="left"
            )
            feat = feat.merge(
                vibe_grp["stance_alignment"].mean().rename("mean_sponsorship_tolerance"),
                on="author_id", how="left"
            )
            # dominant vibe: the single most-frequent room_vibe across that user's engaged posts
            dominant_vibe = (
                vibe_grp["room_vibe"]
                .agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else "neutral")
                .rename("dominant_room_vibe")
            )
            feat = feat.merge(dominant_vibe, on="author_id", how="left")
            feat["dominant_room_vibe"] = feat["dominant_room_vibe"].fillna("neutral").astype(str)

            n_enriched = feat["mean_engaged_consensus"].notna().sum()
            _log.info("room-vibe features attached for %d / %d users", n_enriched, len(feat))
            print(f"  room-vibe enrichment: {n_enriched:,}/{len(feat):,} users have vibe data")
        else:
            # Sentinel columns so downstream cells don't KeyError when vibes are unavailable.
            feat["mean_engaged_consensus"]    = np.nan
            feat["mean_sponsorship_tolerance"] = np.nan
            feat["dominant_room_vibe"]         = "neutral"
            print("  room-vibe enrichment skipped (post_vibes not loaded)")

        return feat


    user_features = build_user_feature_matrix(ig_comments, post_vibes=_post_vibes)
    print(f"User feature matrix built: {len(user_features):,} users.")
    
    # 5. Attach representative comment text from comments_llm (filter to PLATFORM, join by comment_id)
    print(f"Loading comment text from {COMMENTS_LLM_PATH} ...")
    _text_chunks = []
    for _chunk in pd.read_json(COMMENTS_LLM_PATH, lines=True, chunksize=200_000):
        if "platform" in _chunk.columns:
            _chunk = _chunk[_chunk["platform"] == PLATFORM]
        if len(_chunk):
            _text_chunks.append(_chunk[["comment_id", "text"]])
    llm_text = (pd.concat(_text_chunks, ignore_index=True)
                if _text_chunks else pd.DataFrame(columns=["comment_id", "text"]))

    txt = ig_comments[["comment_id", "author_id"]].merge(llm_text, on="comment_id", how="inner")
    top_comments = (
        txt.groupby("author_id").head(20)
           .groupby("author_id")["text"]
           .apply(lambda ts: " ||| ".join(ts.astype(str).tolist()))
           .reset_index()
           .rename(columns={"text": "top_comments_sample"})
    )
    user_features = user_features.merge(top_comments, on="author_id", how="left")
    print(f"Representative comment history attached for {top_comments.shape[0]:,} users.")
    
    # === SAVE CACHE ===
    user_features.to_parquet(USER_FEATURES_CACHE_PATH, index=False)
    print(f"✓ Cached user_features to {USER_FEATURES_CACHE_PATH}")


Binary flags derived.
Sentiment joined: 497,861 comments enriched.
   (no post metadata -> hours_to_comment = NaN)
Loading cached user_features from outputs/user_features_instagram_cache.parquet ...
✓ Cached user_features loaded: 194,513 users


## Feature Selection for Clustering

Below are **all available features** from the comment-level data and user-level aggregations.

In [6]:
if "user_features" not in dir():
    raise RuntimeError("Run the Stage 0 feature-engineering cell first (cell that calls build_user_feature_matrix).")
# ALL AVAILABLE FEATURES FROM STAGE 0
all_available = {
    "Volume & Breadth": [
        "total_comments",
        #"unique_posts_commented",
        #"activity_span_days",
    ],
    "Reply Behavior": [
        "total_replies_made",
        "reply_ratio",
    ],
    # "Timing & Recency": [
    #     "mean_hours_to_comment",
    #     "median_hours_to_comment",
    #     "pct_comments_under_1h",
    #     "pct_comments_under_24h",
    # ],
    "Textual Style": [
        "mean_word_count",
        "mean_mention_count",
        "emoji_usage_rate",
        "question_rate",
        "exclamation_rate",
    ],
    "Concentration": [
        "post_concentration_ratio",
    ],
}

# Display grouped
total_count = 0
for category, features in all_available.items():
    print(f"{category}:")
    for f in features:
        avail = f'✓' if f in user_features.columns else '✗ (missing)'
        print(f"  {avail}  {f}")
        total_count += 1

print(f"Total available: {total_count}")

Volume & Breadth:
  ✓  total_comments
Reply Behavior:
  ✓  total_replies_made
  ✓  reply_ratio
Textual Style:
  ✓  mean_word_count
  ✓  mean_mention_count
  ✓  emoji_usage_rate
  ✓  question_rate
  ✓  exclamation_rate
Concentration:
  ✓  post_concentration_ratio
Total available: 9


In [7]:
if "user_features" not in dir():
    raise RuntimeError("Run the Stage 0 feature-engineering cell first (cell that calls build_user_feature_matrix).")
# EDIT THIS LIST: Remove or comment out features you don't want in clustering
SELECTED_NUMERIC_FEATURES = [
    "total_comments",
    "unique_posts_commented",
    "activity_span_days",
    # "total_replies_made",  # <- REDUNDANT: use reply_ratio instead
    "reply_ratio",
    "mean_hours_to_comment",
    # "median_hours_to_comment",  # <- REDUNDANT: use mean instead
    "pct_comments_under_1h",
    # "pct_comments_under_24h",  # <- OPTIONAL: tight subset of 1h signal
    "mean_word_count",
    "mean_mention_count",
    "emoji_usage_rate",
    "question_rate",
    "exclamation_rate",
    "post_concentration_ratio",
    "mean_sentiment_score",   # average sentiment polarity per user (-1 negative, +1 positive)
    "pct_negative",           # share of comments flagged negative
    "toxicity_rate",          # share of comments flagged toxic (mild or severe)
]

# Filter to only columns that exist
SELECTED_NUMERIC_FEATURES = [c for c in SELECTED_NUMERIC_FEATURES if c in user_features.columns]

print(f"✅ Selected {len(SELECTED_NUMERIC_FEATURES)} numeric features:")
for f in SELECTED_NUMERIC_FEATURES:
    print(f"   • {f}")

✅ Selected 15 numeric features:
   • total_comments
   • unique_posts_commented
   • activity_span_days
   • reply_ratio
   • mean_hours_to_comment
   • pct_comments_under_1h
   • mean_word_count
   • mean_mention_count
   • emoji_usage_rate
   • question_rate
   • exclamation_rate
   • post_concentration_ratio
   • mean_sentiment_score
   • pct_negative
   • toxicity_rate


## Multimodal Media Context

Builds the bridge from a commenter to the **media they engage with**:

1. `user_top_posts` — each `author_id`'s top-N most-commented `media_id`s.
2. media index maps — `media_id → shortcode` and `shortcode → post metadata`.
3. `build_media_manifest()` lists the GCS media tree **once** and groups every `gs://` object by
   shortcode into `images` (jpg/png) and `transcripts` (transcription.txt).
4. `build_user_media_parts(author_id)` emits the multipart payload (metadata text + image `fileData`
   parts + transcript text) that gets attached to that user's Stage 1 / Stage 2 request.

In [8]:
import numpy as np
from google.cloud import storage

if not ATTACH_MEDIA:
    # Non-Instagram platform: no media context — requests are text-only.
    user_top_posts = {}
    media_images, media_transcripts = {}, {}
    def build_user_media_parts(author_id) -> list:
        return []
    print(f"⏭  Media context disabled for platform={PLATFORM}; requests will be text-only.")
else:
    def _norm_id(x):
        """media_id can arrive as int, float (123.0) or str — normalise to a bare digit string."""
        s = str(x)
        return s[:-2] if s.endswith(".0") else s

    # 1) media index maps (bare-numeric media_id matches comments_ml.media_id)
    _mi = media_index[media_index["media_id"].notna()].copy()
    _mi["media_id"] = _mi["media_id"].map(_norm_id)
    _mediaid_to_shortcode = dict(zip(_mi["media_id"], _mi["shortcode"]))
    _meta_by_shortcode    = {r["shortcode"]: r for r in _mi.to_dict("records")}

    # 2) per-user top engaged posts (by comment volume)
    _tp = ig_comments[["author_id", "media_id"]].dropna().copy()
    _tp["author_id"] = _tp["author_id"].astype(str)
    _tp["media_id"]  = _tp["media_id"].map(_norm_id)
    _counts = (
        _tp.groupby(["author_id", "media_id"]).size()
           .reset_index(name="n")
           .sort_values(["author_id", "n"], ascending=[True, False])
    )
    user_top_posts = (
        _counts.groupby("author_id")["media_id"]
               .apply(lambda s: list(s.head(MAX_MEDIA_POSTS_PER_USER)))
               .to_dict()
    )
    print(f"✅ user_top_posts computed for {len(user_top_posts):,} users.")

    # 3) GCS media manifest — one list pass over the whole media tree
    def build_media_manifest(bucket=GCS_BUCKET, prefix=MEDIA_GCS_PREFIX):
        c = storage.Client(project=GCP_PROJECT_ID)
        images, transcripts = {}, {}
        for blob in c.list_blobs(bucket, prefix=prefix):
            name = blob.name
            low  = name.lower()
            segs = name.split("/")          # multimodal_dataset_fixed/<form>/<shortcode>/...
            if len(segs) < 3:
                continue
            sc  = segs[2]
            uri = f"gs://{bucket}/{name}"
            if low.endswith((".jpg", ".jpeg", ".png")):
                images.setdefault(sc, []).append(uri)
            elif low.endswith(".txt") and "transcri" in low:
                transcripts.setdefault(sc, []).append(uri)
        for d in (images, transcripts):
            for k in d:
                d[k].sort()
        print(f"✅ manifest: {len(images):,} posts with images, {len(transcripts):,} with transcripts.")
        return images, transcripts

    media_images, media_transcripts = build_media_manifest()

    # 3b) LOCAL transcript manifest — mirror of the GCS tree at MEDIA_LOCAL_PREFIX.
    #     Reading transcripts off local disk avoids one synchronous GCS download per
    #     post inside the per-user build loop (the real request-building bottleneck).
    import glob as _glob
    media_transcripts_local = {}   # shortcode -> [local .txt paths]
    if os.path.isdir(MEDIA_LOCAL_PREFIX):
        _base = os.path.basename(MEDIA_LOCAL_PREFIX)
        for _p in _glob.iglob(os.path.join(MEDIA_LOCAL_PREFIX, "*", "*", "**", "*.txt"),
                              recursive=True):
            if "transcri" not in os.path.basename(_p).lower():
                continue
            _segs = _p.replace(chr(92), "/").split("/")
            try:                       # <prefix>/<form>/<shortcode>/...
                _sc = _segs[_segs.index(_base) + 2]
            except (ValueError, IndexError):
                continue
            media_transcripts_local.setdefault(_sc, []).append(_p)
        for _k in media_transcripts_local:
            media_transcripts_local[_k].sort()
        print(f"✅ local transcripts: {len(media_transcripts_local):,} posts under {MEDIA_LOCAL_PREFIX}/")
    else:
        print(f"⏭  local media folder {MEDIA_LOCAL_PREFIX!r} not found — using GCS for transcripts.")

    # 4) builders
    _transcript_cache = {}
    def fetch_transcript(sc: str) -> str:
        if sc in _transcript_cache:
            return _transcript_cache[sc]
        chunks = []
        local_paths = media_transcripts_local.get(sc)
        if local_paths:                                   # fast path: local disk, no network
            for _lp in local_paths[:8]:                   # carousel_video has one .txt per slide
                try:
                    with open(_lp, "r", encoding="utf-8", errors="ignore") as _f:
                        chunks.append(_f.read())
                except Exception:
                    pass
        else:                                             # fallback: download from GCS
            c = storage.Client(project=GCP_PROJECT_ID)
            for uri in media_transcripts.get(sc, [])[:8]:
                try:
                    chunks.append(storage.Blob.from_string(uri, client=c).download_as_text())
                except Exception:
                    pass
        txt = " ".join(x.strip() for x in chunks if x.strip())
        _transcript_cache[sc] = txt
        return txt

    def _pick_images(sc: str, k: int):
        """Evenly sample k image URIs across the post (spread frames/slides, not the first k)."""
        uris = media_images.get(sc, [])
        if len(uris) <= k:
            return uris
        step = len(uris) / k
        return [uris[int(i * step)] for i in range(k)]

    def _txt(v):
        return v.strip() if isinstance(v, str) else ""

    def _as_list_str(v):
        if isinstance(v, (list, tuple, np.ndarray)):
            return ", ".join(str(x) for x in v if str(x) not in ("nan", "None", ""))
        return _txt(v)

    def format_post_meta_text(sc: str) -> str:
        row  = _meta_by_shortcode.get(sc, {})
        cf   = _txt(row.get("content_form"))
        bits = [f"POST [{cf or 'post'}] shortcode={sc}"]
        cap  = _txt(row.get("caption"))
        if cap:
            bits.append(f"caption: {cap[:300]}")
        tg = _as_list_str(row.get("tagged_usernames"))
        if tg:
            bits.append(f"tagged: {tg}")
        co = _as_list_str(row.get("coauthors"))
        if co:
            bits.append(f"coauthors: {co}")
        song, artist, atype = _txt(row.get("song_title")), _txt(row.get("artist")), _txt(row.get("audio_type"))
        if song or artist:
            bits.append(f"music: {song} - {artist} ({atype})")
        loc = _txt(row.get("location_name"))
        if loc:
            bits.append(f"location: {loc}")
        return " | ".join(bits)

    def build_user_media_parts(author_id) -> list:
        """Multipart payload for a user's most-engaged posts:
        image/carousel -> photo fileData + metadata; reel/feed/carousel_video -> frame fileData + transcript + metadata."""
        parts = []
        for media_id in user_top_posts.get(str(author_id), []):
            sc = _mediaid_to_shortcode.get(_norm_id(media_id))
            if not sc or sc not in media_images:
                continue
            parts.append({"text": format_post_meta_text(sc)})
            for uri in _pick_images(sc, MAX_IMAGES_PER_POST):
                parts.append({"fileData": {"fileUri": uri, "mimeType": "image/jpeg"}})
            if INCLUDE_TRANSCRIPT:
                t = fetch_transcript(sc)
                if t:
                    parts.append({"text": "transcript: " + t[:MAX_TRANSCRIPT_CHARS]})
        return parts

    _demo = next((a for a in user_features["author_id"].astype(str) if build_user_media_parts(a)), None)
    print("✅ builders ready. Example user media parts:",
          len(build_user_media_parts(_demo)) if _demo else 0)

⏭  Media context disabled for platform=instagram; requests will be text-only.


d:\conda_envs\ma_env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## Batch Infrastructure

The five reusable stages of a Vertex batch job, lifted from `Vertex_Batch_Inference.ipynb`:
**write JSONL → upload → submit → poll → retrieve+parse**. Submit is non-blocking; we re-GET the job
each poll until a terminal state; outputs may be sharded across files so we iterate.

In [9]:
import json, os, time, datetime, re
from google.cloud import storage

def strip_fences(s: str) -> str:
    s = s.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    return s.strip()

def write_jsonl(lines: list, path: str) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for obj in lines:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
    print(f"[prep] wrote {len(lines):,} requests -> {path}")
    return path

def upload_to_gcs(local_path: str, bucket_name: str, blob_name: str) -> str:
    c = storage.Client(project=GCP_PROJECT_ID)
    c.bucket(bucket_name).blob(blob_name).upload_from_filename(local_path)
    uri = f"gs://{bucket_name}/{blob_name}"
    print(f"[upload] {local_path} -> {uri}")
    return uri

def submit_batch_job(input_uri: str, output_uri: str, model: str):
    """Non-blocking submit. dest is a PREFIX; Vertex writes a unique run subfolder under it."""
    job = client.batches.create(
        model=model, src=input_uri,
        config=CreateBatchJobConfig(dest=output_uri),
    )
    print(f"[submit] {model} -> {job.name}  ({job.state})")
    return job

def poll_until_complete(job_name: str):
    terminal = {JobState.JOB_STATE_SUCCEEDED, JobState.JOB_STATE_FAILED,
                JobState.JOB_STATE_CANCELLED, JobState.JOB_STATE_PAUSED}
    while True:
        job = client.batches.get(name=job_name)
        print(f"[poll {datetime.datetime.now():%H:%M:%S}] state = {job.state}")
        if job.state in terminal:
            return job
        time.sleep(POLL_INTERVAL_SECONDS)

def retrieve_response_texts(job, bucket_name: str) -> list:
    """Download every output shard and return the model's text per row (None for failed rows)."""
    out_loc = job.dest.gcs_uri
    prefix  = out_loc.replace(f"gs://{bucket_name}/", "")
    c  = storage.Client(project=GCP_PROJECT_ID)
    bk = c.bucket(bucket_name)
    blobs = [b for b in bk.list_blobs(prefix=prefix) if b.name.endswith(".jsonl")]
    texts = []
    for blob in blobs:
        for line in blob.download_as_text().splitlines():
            if not line.strip():
                continue
            rec  = json.loads(line)
            resp = rec.get("response")
            if not resp:
                texts.append(None)               # row-level failure / safety block
                continue
            try:
                parts = resp["candidates"][0]["content"]["parts"]
                texts.append("".join(p.get("text", "") for p in parts))
            except (KeyError, IndexError):
                texts.append(None)
    print(f"[retrieve] {len(texts):,} response rows from {len(blobs)} shard(s)")
    return texts

def record_batch_job(tag: str, job, dest: str) -> dict:
    """Persist a submitted job so a LATER run (even a fresh kernel) can retrieve it."""
    rec = {"name": job.name, "dest": dest, "state": str(job.state),
           "submitted_at": datetime.datetime.now().isoformat()}
    with open(f"{LOCAL_DIR}/{tag}_job.json", "w", encoding="utf-8") as f:
        json.dump(rec, f, indent=2)
    print(f"[record] {tag} job -> {LOCAL_DIR}/{tag}_job.json")
    return rec

def get_recorded_job(tag: str):
    """Re-fetch the live job for a recorded submission (raises if none recorded)."""
    p = f"{LOCAL_DIR}/{tag}_job.json"
    if not os.path.exists(p):
        raise FileNotFoundError(f"No submitted '{tag}' job recorded at {p} — submit it first.")
    with open(p, encoding="utf-8") as f:
        rec = json.load(f)
    job = client.batches.get(name=rec["name"])
    print(f"[{tag}] {rec['name']} -> {job.state}")
    return job

print("✅ Batch infrastructure ready.")

✅ Batch infrastructure ready.


## Stage 1 — Taxonomy Discovery (Batch · Multimodal)

Groups of `STAGE1_USERS_PER_REQUEST` sampled users go into each JSONL line, every user carrying their
engaged-post media. Flash returns candidate archetypes per line; we concatenate all candidates and
write `taxonomy.json` for **human review** (consolidate → set `status="APPROVED"`).

In [10]:
from tqdm import tqdm

# defaults for pb-config vars used here (overridden if pb-config ran first)
if "STRATIFY_BY_SENTIMENT" not in dir(): STRATIFY_BY_SENTIMENT = True
if "SENTIMENT_STRATA_COL"  not in dir(): SENTIMENT_STRATA_COL  = "sentiment"
if "SENTIMENT_PATH"        not in dir(): SENTIMENT_PATH        = f"{LOCAL_DIR}/sentiment_{PLATFORM}.parquet"

# sentinel: define canon_sentiment here too so this cell is self-contained
_SENT_CANON_MAP_S1 = {
    "positive": "positive", "negative": "negative", "neutral": "neutral",
    "amused": "positive", "amusement": "positive", "joke": "positive",
    "joy": "positive", "support": "positive", "supportive": "positive",
    "affection": "positive", "anticipation": "positive",
    "sarcastic": "negative", "sarcasm": "negative", "sadness": "negative",
    "question": "neutral", "suggestion": "neutral", "surprise": "neutral",
    "mixed": "neutral", "spam": "neutral",
}
if "canon_sentiment" not in dir():
    def canon_sentiment(series):
        s = series.astype(str).str.strip().str.lower()
        return s.map(_SENT_CANON_MAP_S1).fillna("neutral")


STAGE1_SYSTEM_PROMPT = (
    "You are an expert community analyst for a major Italian influencer agency.\n"
    "Identify distinct audience persona archetypes from Instagram commenter behaviour.\n"
    "Each user profile carries quantitative metrics, a sample of their comments, AND the media\n"
    "(images / video frames + transcript) of the posts they engage with most.\n"
    "Use BOTH how they comment and WHAT content they engage with.\n"
    "For each recurring pattern output a candidate persona with: a short codename (e.g. SUPERFAN),\n"
    "a 1-sentence behavioural description, key quantitative signals, and 2-3 verbatim comment fragments.\n"
    "Output ONLY a valid JSON array. No preamble, no markdown fences.\n"
    'Schema: [{"codename": str, "description": str, "signals": [str], "examples": [str]}]'
)

# Stage 1 comment-history budget: show WHOLE comments per user, but bounded.
STAGE1_MAX_COMMENTS_PER_USER = 10    # at most N whole comments per user
STAGE1_MAX_CHARS_PER_COMMENT = 200   # truncate any single very long comment
STAGE1_MAX_CHARS_PER_USER    = 800   # overall ceiling per user (token guard)
def _fmt_user_comments(raw):
    parts = [p.strip() for p in str(raw).split("|||") if p.strip()]
    out, used = [], 0
    for p in parts[:STAGE1_MAX_COMMENTS_PER_USER]:
        p = p[:STAGE1_MAX_CHARS_PER_COMMENT]
        if used + len(p) > STAGE1_MAX_CHARS_PER_USER:
            break
        out.append(p)
        used += len(p)
    return " || ".join(f'"{p}"' for p in out) if out else "(none)"

def format_user_profile_for_stage1(row) -> str:
    return (
        f"USER: {row['author_id']} | "
        f"Total comments: {int(row['total_comments'])} | "
        f"Unique posts: {int(row['unique_posts_commented'])} | "
        f"Activity span: {int(row['activity_span_days'])} days | "
        f"Avg hrs to comment: {row['mean_hours_to_comment']:.1f}h | "
        f"Early commenter (<1h): {row['pct_comments_under_1h']:.0%} | "
        f"Reply ratio: {row['reply_ratio']:.0%} | "
        f"Avg word count: {row['mean_word_count']:.0f} | "
        f"Emoji rate: {row['emoji_usage_rate']:.0%} | "
        f"Question rate: {row['question_rate']:.0%} | "
        f"Comment history (up to {STAGE1_MAX_COMMENTS_PER_USER} whole comments): "
        f"{_fmt_user_comments(row.get('top_comments_sample', ''))}"
    )

def build_stage1_line(group_df) -> dict:
    parts = [{"text": STAGE1_SYSTEM_PROMPT + "\n\n--- USER PROFILES BATCH (with engaged-post media) ---"}]
    for _, row in group_df.iterrows():
        parts.append({"text": "\n" + format_user_profile_for_stage1(row)})
        parts.extend(build_user_media_parts(row["author_id"]))
    parts.append({"text": "\nIdentify all distinct behavioural archetypes in this batch. Output ONLY the JSON array."})
    return {"request": {"contents": [{"role": "user", "parts": parts}],
                        "generationConfig": gen_config_dict(MAX_OUTPUT_TOKENS_STAGE1, THINKING_BUDGET_STAGE1)}}

def save_taxonomy_for_review(candidates, output_path=TAXONOMY_JSON_PATH):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    out = {
        "status": "PENDING_HUMAN_REVIEW",
        "instructions": ("Consolidate the candidates into a MECE taxonomy. Each final persona needs a "
                         "unique 'codename' (plus 'label', 'description', 'quantitative_signals', "
                         "'example_comments'). Set status='APPROVED' before Stage 2."),
        "raw_candidates": candidates,
        "final_taxonomy": [],
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"✅ Raw candidates saved -> {output_path}")
    print("   ⚠️  HUMAN ACTION: review, consolidate, set status='APPROVED'.")

def stratified_user_sample(user_features_df, n_sample=SAMPLE_N_USERS, seed=SAMPLE_SEED):
    """USER-level stratified sample by dominant sentiment_cat.

    Each author is assigned ONE stratum = their modal comment sentiment_cat, then
    users are drawn PROPORTIONALLY across {positive, negative, neutral} up to n_sample.
    Degrades to a plain random sample if STRATIFY_BY_SENTIMENT is off or the sentiment
    file is missing.
    """
    if not STRATIFY_BY_SENTIMENT or not os.path.exists(SENTIMENT_PATH):
        if STRATIFY_BY_SENTIMENT:
            print(f"⚠️  {SENTIMENT_PATH} not found — falling back to PLAIN RANDOM sample.")
        return user_features_df.sample(n=min(n_sample, len(user_features_df)),
                                       random_state=seed).reset_index(drop=True)

    sent = pd.read_parquet(SENTIMENT_PATH)
    if SENTIMENT_STRATA_COL in sent.columns:
        sent[SENTIMENT_STRATA_COL] = canon_sentiment(sent[SENTIMENT_STRATA_COL])
    if "author_id" in sent.columns:
        sa = sent[["author_id", SENTIMENT_STRATA_COL]].dropna().copy()
    else:  # fallback: join comment-level sentiment onto ig_comments to recover author_id
        sa = (sent[["comment_id", SENTIMENT_STRATA_COL]].dropna()
              .merge(ig_comments[["comment_id", "author_id"]], on="comment_id", how="inner"))
    sa["author_id"] = sa["author_id"].astype(str)

    # one stratum per author = their MODAL (dominant) comment sentiment_cat
    author_strata = (sa.groupby("author_id")[SENTIMENT_STRATA_COL]
                       .agg(lambda x: x.value_counts().idxmax())
                       .rename("author_sentiment"))

    uf = user_features_df.copy()
    uf["author_id"] = uf["author_id"].astype(str)
    uf = uf.merge(author_strata, on="author_id", how="left")
    uf["author_sentiment"] = uf["author_sentiment"].fillna("neutral")

    n = min(n_sample, len(uf))
    frac = n / len(uf) if len(uf) else 0.0
    parts = []
    for cat, grp in uf.groupby("author_sentiment"):
        k = max(1, int(round(len(grp) * frac)))
        parts.append(grp.sample(n=min(k, len(grp)), random_state=seed))
    out = (pd.concat(parts)
             .sample(frac=1.0, random_state=seed)   # shuffle so groups mix sentiments
             .head(n).reset_index(drop=True))
    print(f"✅ Stratified Stage-1 sample (user-level dominant {SENTIMENT_STRATA_COL}): "
          f"{len(out):,} users | strata mix (%):")
    print(out["author_sentiment"].value_counts(normalize=True).mul(100).round(1).to_string())
    return out


# ── SUBMIT (returns immediately — safe to close the laptop) ────────────────────
def _estimate_tokens(lines: list) -> int:
    """Rough token estimate: ~4 chars per token across all request JSON."""
    total_chars = sum(len(json.dumps(l, ensure_ascii=False)) for l in lines)
    return total_chars // 4

def submit_stage1(user_features_df, n_sample=SAMPLE_N_USERS,
                  group_size=STAGE1_USERS_PER_REQUEST, seed=SAMPLE_SEED):
    print(f"\n{'='*60}\nSTAGE 1 (batch submit) — Taxonomy Discovery\n"
          f"  Sample: {n_sample} users | {group_size}/request | Model: {MODEL_STAGE1_EXPLORATORY}\n{'='*60}")
    sample_df = stratified_user_sample(user_features_df, n_sample=n_sample, seed=seed)
    # Persist the sampled author set so Pathway B clusters the IDENTICAL users.
    sample_df[["author_id"]].to_parquet(
        f"{LOCAL_DIR}/stage1_sample_users_{PLATFORM}.parquet", index=False)
    groups = [sample_df.iloc[i:i+group_size] for i in range(0, len(sample_df), group_size)]
    lines  = [build_stage1_line(g) for g in tqdm(groups, desc="Build Stage 1 requests")]

    # Split into chunks that fit under Vertex AI 1M token limit.
    chunks, chunk, chunk_tokens = [], [], 0
    for line in lines:
        tok = len(json.dumps(line, ensure_ascii=False)) // 4
        if chunk and chunk_tokens + tok > MAX_BATCH_TOKENS:
            chunks.append(chunk)
            chunk, chunk_tokens = [], 0
        chunk.append(line)
        chunk_tokens += tok
    if chunk:
        chunks.append(chunk)

    print(f"  Splitting into {len(chunks)} batch job(s) to stay under {MAX_BATCH_TOKENS:,} tokens each.")
    jobs = []
    for idx, chunk_lines in enumerate(chunks):
        tag = f"stage1_chunk{idx}"
        local_path = f"{LOCAL_DIR}/stage1_input_chunk{idx}.jsonl"
        in_uri  = upload_to_gcs(write_jsonl(chunk_lines, local_path),
                                GCS_BUCKET, BATCH_INPUT_PREFIX + f"stage1_input_chunk{idx}.jsonl")
        out_uri = f"gs://{GCS_BUCKET}/{BATCH_OUTPUT_PREFIX}stage1_chunk{idx}/"
        job = submit_batch_job(in_uri, out_uri, MODEL_STAGE1_EXPLORATORY)
        record_batch_job(tag, job, out_uri)
        jobs.append(job)

    # Also write a manifest so retrieve_stage1() knows how many chunks to expect.
    with open(f"{LOCAL_DIR}/stage1_chunks.json", "w") as f:
        json.dump({"n_chunks": len(chunks), "tags": [f"stage1_chunk{i}" for i in range(len(chunks))]}, f)
    print(f"\n\U0001f4e4 Stage 1 submitted ({len(chunks)} job(s), {len(lines)} total requests). Safe to close the laptop.")
    print("   When all finish, run:  retrieve_stage1()   -> writes taxonomy.json for review")
    return jobs

# ── RETRIEVE (run later, even from a fresh kernel) ────────────────────────────
def _salvage_json_objects(txt):
    """Extract complete top-level JSON objects from a truncated array string."""
    out, depth, start, instr, esc = [], 0, None, False, False
    for i, ch in enumerate(txt):
        if instr:
            if esc:        esc = False
            elif ch == "\\": esc = True
            elif ch == '"': instr = False
            continue
        if ch == '"':
            instr = True
        elif ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                try:
                    out.append(json.loads(txt[start:i+1]))
                except Exception:
                    pass
                start = None
    return [o for o in out if isinstance(o, dict)]

def retrieve_stage1(save=True):
    manifest_path = f"{LOCAL_DIR}/stage1_chunks.json"
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            manifest = json.load(f)
        tags = manifest["tags"]
        print(f"[retrieve] manifest found: {len(tags)} chunk(s) -> {tags}")
    else:
        print(f"[retrieve] no manifest at {manifest_path!r} — falling back to legacy single job")
        tags = ["stage1"]  # legacy single-job path

    all_texts = []
    for tag in tags:
        job = get_recorded_job(tag)
        if job.state != JobState.JOB_STATE_SUCCEEDED:
            print(f"⏳ {tag} not ready (state={job.state}). Re-run retrieve_stage1() later.")
            return None
        all_texts.extend(retrieve_response_texts(job, GCS_BUCKET))
    print(f"[retrieve] {len(all_texts):,} total response rows from {len(tags)} job(s)")

    candidates = []
    _truncated = 0
    for t in all_texts:
        if not t:
            continue
        txt = strip_fences(t)
        try:
            c = json.loads(txt)
            if isinstance(c, list):
                candidates.extend(x for x in c if isinstance(x, dict))
            elif isinstance(c, dict):
                candidates.append(c)
        except Exception:
            # Salvage: model output truncated at max_output_tokens mid-array.
            # Recover every complete top-level {...} object we can.
            salvaged = _salvage_json_objects(txt)
            if salvaged:
                candidates.extend(salvaged)
                _truncated += 1
    if _truncated:
        print(f"   recovered partial objects from {_truncated} truncated response(s) "
              f"(consider raising MAX_OUTPUT_TOKENS_STAGE1 further).")
    print(f"✅ Stage 1 complete. Raw candidates: {len(candidates)}")
    if save:
        save_taxonomy_for_review(candidates)
    return candidates

print("✅ Stage 1 functions ready (submit_stage1 / retrieve_stage1).")

✅ Stage 1 functions ready (submit_stage1 / retrieve_stage1).


### Stage 1 — Inspect Taxonomy Results

In [11]:
if os.path.exists(TAXONOMY_JSON_PATH):
    with open(TAXONOMY_JSON_PATH, "r", encoding="utf-8") as f:
        taxonomy_data = json.load(f)
    status = taxonomy_data.get("status")
    print(f"Taxonomy status: {status}")
    if status == "APPROVED" and taxonomy_data.get("final_taxonomy"):
        display(pd.DataFrame(taxonomy_data["final_taxonomy"]))
    elif taxonomy_data.get("raw_candidates"):
        print("Showing raw candidates (pending review):")
        display(pd.DataFrame(taxonomy_data["raw_candidates"]))
    else:
        print("No candidates found — run Stage 1 first.")
else:
    print("taxonomy.json not found — run Stage 1 first.")

Taxonomy status: PENDING_HUMAN_REVIEW
Showing raw candidates (pending review):


,codename,description,signals,examples
0,EMOJI_REACTOR,This user primarily expresses appreciation or ...,"[Avg word count: 0-2, Emoji rate: 100%, Reply ...","[👏👏👏😂, 😂]"
1,CASUAL_COMMENTER,"This user engages occasionally with short, rel...","[Total comments: 1-2, Avg word count: 2-5, Emo...","[@camihawke andando dal dentista forse 🤣, @vio..."
2,ENGAGED_REPLIER,This user actively participates in conversatio...,"[Reply ratio: 100%, Avg word count: >10, Emoji...",[@silviacerigioni l ho quasi imparata occhiooo...
3,EARLY_APPRECIATOR,"This user is quick to react to new content, of...","[Early commenter (<1h): >50%, Avg word count: ...","[@sophiadametto 😍😍😍😍, TGICami]"
4,THE ENTHUSIAST,This persona expresses strong positive emotion...,"[High emoji rate (often 0% or 100%), Tags othe...","[Dio fantiffimoooo @sr08__, Ma quanto è bella?..."
...,...,...,...,...
19713,DEVOTED_FOLLOWER,"This archetype shows consistent, long-term eng...","[Activity span: 319 days (long-term), Avg word...","[❤️❤️❤️❤️❤️❤️❤️❤️❤️, Camiiiiii ma dov’è Tonino..."
19714,CASUAL_APPRECIATOR,"This user offers brief, positive affirmation, ...","[Total comments: 1 (minimal engagement), Avg w...",[❤️]
19715,ADMIRING_FAN,"This archetype expresses direct compliments, f...","[Total comments: 1 (single interaction), Avg w...","[Non sei bella, di più.♥️]"
19716,THE ADMIRER,This persona consistently expresses positive s...,"[High emoji rate (83%), Very low average word ...","[♥️, Che bonaggine, Bella Cami❤️]"


In [12]:
import pandas as pd, os

def show_sample_characteristics(platform=PLATFORM, local_dir=LOCAL_DIR,
                                sentiment_path=SENTIMENT_PATH,
                                strata_col=SENTIMENT_STRATA_COL):
    """Print size, sentiment distribution, and key behavioural stats for the Stage-1 sample."""
    sample_path = f"{local_dir}/stage1_sample_users_{platform}.parquet"
    if not os.path.exists(sample_path):
        print(f"sample file not found: {sample_path} — run submit_stage1() first.")
        return

    sample_ids = pd.read_parquet(sample_path)["author_id"].astype(str)
    n_sample = len(sample_ids)

    # Behavioural features for the sampled users
    uf = user_features.copy()
    uf["author_id"] = uf["author_id"].astype(str)
    sample_uf = uf[uf["author_id"].isin(set(sample_ids))].copy()

    print(f"=== Stage-1 Sample Characteristics ({platform}) ===")
    print(f"  Total users in sample : {n_sample:,}")
    print(f"  Total users in pool   : {len(uf):,}")
    print(f"  Sample fraction       : {n_sample / len(uf):.1%}")

    # --- Sentiment strata distribution ---
    if STRATIFY_BY_SENTIMENT and os.path.exists(sentiment_path):
        sent = pd.read_parquet(sentiment_path)
        if "author_id" not in sent.columns:
            sent = (sent[["comment_id", strata_col]].dropna()
                    .merge(ig_comments[["comment_id", "author_id"]], on="comment_id", how="inner"))
        sent["author_id"] = sent["author_id"].astype(str)
        author_dom = (sent.groupby("author_id")[strata_col]
                         .agg(lambda x: x.value_counts().idxmax())
                         .rename("dominant_sentiment"))
        sample_sent = author_dom[author_dom.index.isin(set(sample_ids))]
        pool_sent   = author_dom

        counts  = sample_sent.value_counts().sort_index()
        pct     = sample_sent.value_counts(normalize=True).mul(100).round(1).sort_index()
        pool_pct = pool_sent.value_counts(normalize=True).mul(100).round(1).sort_index()

        print(f"Sentiment strata (user-level dominant {strata_col}):")
        print(f"  {'Category':<12} {'Sample n':<10} {'Sample %':<12} {'Pool %'}")
        print(f"  {'-'*46}")
        for cat in sorted(set(counts.index) | set(pool_pct.index)):
            n   = counts.get(cat, 0)
            sp  = pct.get(cat, 0.0)
            pp  = pool_pct.get(cat, 0.0)
            print(f"  {cat:<12} {n:<10} {sp:<12.1f} {pp:.1f}")
    else:
        print("  Sentiment stratification off (or sentiment file missing).")

    # --- Key behavioural stats ---
    numeric_cols = [c for c in [
        "total_comments", "unique_posts_commented", "activity_span_days",
        "mean_hours_to_comment", "pct_comments_under_1h", "reply_ratio",
        "mean_word_count", "emoji_usage_rate", "question_rate"
    ] if c in sample_uf.columns]

    if numeric_cols:
        stats = sample_uf[numeric_cols].describe().loc[["mean", "50%", "std"]]
        stats.index = ["mean", "median", "std"]
        print(f"Key behavioural stats (sample):")
        print(stats.to_string(float_format="{:.2f}".format))

    # --- dominant_room_vibe breakdown ---
    if "dominant_room_vibe" in sample_uf.columns:
        vibe_counts = sample_uf["dominant_room_vibe"].value_counts()
        print(f"Dominant room vibe distribution:")
        for v, n in vibe_counts.items():
            bar = "#" * int(n / vibe_counts.max() * 20)
            print(f"  {v:<20} {n:>4}  {bar}")

    return sample_uf


show_sample_characteristics()


NameError: name 'SENTIMENT_STRATA_COL' is not defined

## Pathway B — UMAP + HDBSCAN Clustering -> Taxonomy (alternative to LLM Consolidation)

This is an **alternative** to the LLM-consolidation cell above. Run **EITHER** Pathway A
(`llm_consolidate_taxonomy`) **OR** this block — not both. Pathway B clusters the **same
stratified Stage-1 sample** by behaviour (UMAP + HDBSCAN), asks Gemini to name each cluster,
and writes the result to `taxonomy.json -> final_taxonomy` in the **identical 5-key schema**
Stage 2 consumes. Set `CONSOLIDATION_PATHWAY = "B_CLUSTER"` so the Pathway-A wrapper cell
does not overwrite this taxonomy. Then review and set `status="APPROVED"`.

In [ ]:
# Pathway B dependencies (run once)
!pip install -q umap-learn hdbscan scikit-learn


In [ ]:
# Pathway B config — UMAP + HDBSCAN knobs (tuned for the smaller Stage-1 sample).
# Names match the legacy Stage 3 config; harmless if that block runs later (just re-set).
UMAP_N_NEIGHBORS          = 30
UMAP_MIN_DIST             = 0.0
UMAP_N_COMPONENTS_CLUSTER = 15
UMAP_METRIC               = "euclidean"
UMAP_RANDOM_STATE         = 42

HDBSCAN_MIN_SAMPLES       = 10
HDBSCAN_CLUSTER_SELECTION = "eom"
HDBSCAN_CLUSTER_EPSILON   = 0.0

MODEL_STAGE3_NAMING          = "gemini-2.5-flash-lite"
MAX_SAMPLE_USERS_PER_CLUSTER = 50
PATHWAY_B_TARGET_CLUSTERS = 12
TARGET_PERSONAS            = PATHWAY_B_TARGET_CLUSTERS
STRATIFY_BY_SENTIMENT      = True
SENTIMENT_STRATA_COL       = "sentiment"   # column in sentiment_<platform>.parquet to stratify on
CLUSTER_PERSONA_MAP_PATH   = f"{LOCAL_DIR}/cluster_persona_map_{PLATFORM}.json"
print("Pathway B config loaded (target ~", PATHWAY_B_TARGET_CLUSTERS, "clusters).")


Pathway B config loaded (target ~ 12 clusters).


In [ ]:
# Re-root the clustering on the STRATIFIED STAGE-1 SAMPLE users (NOT Stage 2 output).
import os, pandas as pd, numpy as np

_sample_path = f"{LOCAL_DIR}/stage1_sample_users_{PLATFORM}.parquet"
if os.path.exists(_sample_path):
    _sample_ids = set(pd.read_parquet(_sample_path)["author_id"].astype(str))
    pb_df = user_features[user_features["author_id"].astype(str).isin(_sample_ids)].copy()
    print(f"Loaded Stage-1 sample users from {_sample_path}")
else:
    print("⚠️  stage1 sample file missing — drawing a fresh stratified sample live.")
    pb_df = stratified_user_sample(user_features)
pb_df = pb_df.reset_index(drop=True)
print(f"Pathway B working set: {len(pb_df):,} users (the Stage-1 stratified sample).")


Loaded Stage-1 sample users from outputs/stage1_sample_users_instagram.parquet
Pathway B working set: 20,000 users (the Stage-1 stratified sample).


In [ ]:
# Feature matrix: standardised numeric features + OHE dominant_room_vibe.
# (No persona_codename OHE — Stage 2 labels don't exist yet; this BUILDS the taxonomy.)
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC_FEATURES = SELECTED_NUMERIC_FEATURES.copy()
print(f"Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")

num_data = pb_df[NUMERIC_FEATURES].fillna(pb_df[NUMERIC_FEATURES].median())
X_num = StandardScaler().fit_transform(num_data)

if "dominant_room_vibe" in pb_df.columns and pb_df["dominant_room_vibe"].nunique() > 1:
    ohe_vibe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_vibe = ohe_vibe.fit_transform(pb_df[["dominant_room_vibe"]])
    print(f"OHE dominant_room_vibe dims: {X_vibe.shape[1]} -> {ohe_vibe.categories_[0].tolist()}")
    X = np.hstack([X_num, X_vibe])
else:
    print("dominant_room_vibe: single value or absent — numeric-only matrix.")
    X = X_num
print(f"✅ Pathway B feature matrix: {X.shape}  (rows=sample users, cols=numeric+OHE)")


Numeric features (15): ['total_comments', 'unique_posts_commented', 'activity_span_days', 'reply_ratio', 'mean_hours_to_comment', 'pct_comments_under_1h', 'mean_word_count', 'mean_mention_count', 'emoji_usage_rate', 'question_rate', 'exclamation_rate', 'post_concentration_ratio', 'mean_sentiment_score', 'pct_negative', 'toxicity_rate']
OHE dominant_room_vibe dims: 1382 -> ['-0.0005494505494505475', '-0.011764705882352943', '-0.03640776699029126', '-0.05456919060052219', '-0.07350746268656717', '-0.08333333333333333', '-0.08702702702702701', '-0.09218009478672985', '-0.13157894736842105', '-0.19411764705882353', '-0.23809523809523808', '-0.5468468468468469', '0.013833333333333312', '0.0700854700854701', '0.10051020408163265', '0.10789473684210522', '0.11714285714285716', '0.11730769230769231', '0.13484848484848483', '0.1349056603773585', '0.1359375', '0.13636363636363638', '0.14144144144144144', '0.15189873417721522', '0.15934065934065933', '0.16140065146579807', '0.16428571428571428', 

In [ ]:
# UMAP -> 15-D embedding for HDBSCAN.
import umap

print(f"Running UMAP ({UMAP_N_COMPONENTS_CLUSTER}-D) on {len(X):,} sample users ...")
reducer_cluster = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=UMAP_N_COMPONENTS_CLUSTER,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
    low_memory=True,
    verbose=True,
)
X_umap_cluster = reducer_cluster.fit_transform(X)
print(f"✅ UMAP embedding: {X_umap_cluster.shape}")


Running UMAP (15-D) on 20,000 sample users ...
UMAP(min_dist=0.0, n_components=15, n_jobs=1, n_neighbors=30, random_state=42, verbose=True)
Thu Jun 11 13:46:36 2026 Construct fuzzy simplicial set
Thu Jun 11 13:46:36 2026 Finding Nearest Neighbors


d:\conda_envs\ma_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Thu Jun 11 13:46:36 2026 Building RP forest with 12 trees
Thu Jun 11 13:46:37 2026 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	 4  /  14
	Stopping threshold met -- exiting after 4 iterations
Thu Jun 11 13:46:43 2026 Finished Nearest Neighbor Search
Thu Jun 11 13:46:43 2026 Construct embedding


Epochs completed:   4%| ▎          7/200 [00:00]

	completed  0  /  200 epochs


...

	completed  160  /  200 epochs


Epochs completed:  92%| █████████▏ 184/200 [00:18]

	completed  180  /  200 epochs


Epochs completed: 100%| ██████████ 200/200 [00:20]

Thu Jun 11 13:47:05 2026 Finished embedding
✅ UMAP embedding: (20000, 15)


In [ ]:
# HDBSCAN auto-tuned toward PATHWAY_B_TARGET_CLUSTERS (soft <= 12).
import hdbscan, numpy as np

def cluster_near_target(emb, target=PATHWAY_B_TARGET_CLUSTERS):
    best = None
    for frac in (0.005, 0.01, 0.02, 0.03, 0.05, 0.08):
        mcs = max(5, int(len(emb) * frac))
        lab = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=HDBSCAN_MIN_SAMPLES,
            cluster_selection_method=HDBSCAN_CLUSTER_SELECTION,
            cluster_selection_epsilon=HDBSCAN_CLUSTER_EPSILON,
        ).fit_predict(emb)
        k = len(set(lab[lab >= 0]))
        if best is None or abs(k - target) < abs(best[1] - target):
            best = (mcs, k, lab)
        if k <= target:
            break
    return best

pb_mcs, pb_k, pb_labels = cluster_near_target(X_umap_cluster)
pb_df["macro_cluster"] = pb_labels
pb_unique_clusters = sorted(set(pb_labels[pb_labels >= 0]))
pb_n_noise = int((pb_labels == -1).sum())
print(f"Pathway B: min_cluster_size={pb_mcs} -> {pb_k} clusters | "
      f"noise={pb_n_noise:,} ({100*pb_n_noise/len(pb_labels):.1f}%)")
for c in pb_unique_clusters:
    cnt = int((pb_labels == c).sum())
    print(f"   Cluster {c:>2}: {cnt:>6,} users ({100*cnt/len(pb_labels):.1f}%)")
# Include noise as a named class so summaries and LLM see it.
if pb_n_noise > 0:
    pb_unique_clusters_with_noise = [-1] + pb_unique_clusters
    print(f"   Cluster -1 (NOISE): {pb_n_noise:>6,} users ({100*pb_n_noise/len(pb_labels):.1f}%)")
else:
    pb_unique_clusters_with_noise = pb_unique_clusters


Pathway B: min_cluster_size=600 -> 11 clusters | noise=1,169 (5.8%)
   Cluster  0:  1,039 users (5.2%)
   Cluster  1:  1,587 users (7.9%)
   Cluster  2:    942 users (4.7%)
   Cluster  3:    824 users (4.1%)
   Cluster  4:  1,183 users (5.9%)
   Cluster  5:    756 users (3.8%)
   Cluster  6:    817 users (4.1%)
   Cluster  7:    893 users (4.5%)
   Cluster  8:  7,120 users (35.6%)
   Cluster  9:    729 users (3.6%)
   Cluster 10:  2,941 users (14.7%)
   Cluster -1 (NOISE):  1,169 users (5.8%)


In [ ]:
# 3-D UMAP visualisation of Pathway B clusters.
# Reduces the 15-D clustering embedding to 3 dimensions for scatter plotting.
import umap, numpy as np, pandas as pd
import plotly.express as px

print("Running 3-D UMAP for visualisation ...")
reducer_viz3 = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=0.1,
    n_components=3,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
    low_memory=True,
    verbose=False,
)
X_umap3 = reducer_viz3.fit_transform(X_umap_cluster)

viz_df = pd.DataFrame({
    "x": X_umap3[:, 0],
    "y": X_umap3[:, 1],
    "z": X_umap3[:, 2],
    "cluster": pb_labels.astype(str),
    "is_noise": pb_labels == -1,
})
# Label noise distinctly so it stands out in the legend.
viz_df["cluster_label"] = viz_df["cluster"].apply(
    lambda c: "NOISE" if c == "-1" else f"Cluster {c}"
)

# Colour map: noise is grey, real clusters get distinct colours.
unique_labels = sorted(viz_df["cluster_label"].unique(), key=lambda x: (x == "NOISE", x))
color_map = {}
palette = px.colors.qualitative.Bold + px.colors.qualitative.Pastel
ci = 0
for lbl in unique_labels:
    if lbl == "NOISE":
        color_map[lbl] = "#aaaaaa"
    else:
        color_map[lbl] = palette[ci % len(palette)]
        ci += 1

fig = px.scatter_3d(
    viz_df, x="x", y="y", z="z",
    color="cluster_label",
    color_discrete_map=color_map,
    opacity=0.5,
    size_max=3,
    title=f"Pathway B — UMAP 3-D ({pb_k} clusters + noise)",
    labels={"x": "UMAP-1", "y": "UMAP-2", "z": "UMAP-3", "cluster_label": "Cluster"},
    category_orders={"cluster_label": unique_labels},
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(legend=dict(itemsizing="constant"), margin=dict(l=0, r=0, t=40, b=0))
fig.show()

# Cluster size bar chart.
size_df = (
    viz_df.groupby("cluster_label", observed=True)
    .size().reset_index(name="n_users")
    .sort_values("n_users", ascending=False)
)
fig2 = px.bar(
    size_df, x="cluster_label", y="n_users",
    color="cluster_label",
    color_discrete_map=color_map,
    title="Cluster sizes (Pathway B)",
    labels={"cluster_label": "Cluster", "n_users": "Users"},
    category_orders={"cluster_label": unique_labels},
)
fig2.update_layout(showlegend=False)
fig2.show()


Running 3-D UMAP for visualisation ...


d:\conda_envs\ma_env\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
# Per-cluster summary (behaviour + room-vibe mix + sample comments) for LLM naming.
# cluster_id == -1 is the HDBSCAN noise class.

# Build once: per-user dominant comment sentiment, used to stratify which users we show the LLM.
_USER_DOM_SENT = None
try:
    if os.path.exists(SENTIMENT_PATH):
        _su = pd.read_parquet(SENTIMENT_PATH)
        if "author_id" not in _su.columns and "comment_id" in _su.columns:
            _su = _su.merge(ig_comments[["comment_id", "author_id"]], on="comment_id", how="left")
        if {"author_id", "sentiment"}.issubset(_su.columns):
            _su["author_id"] = _su["author_id"].astype(str)
            _su["sentiment"] = canon_sentiment(_su["sentiment"])
            _USER_DOM_SENT = (_su.dropna(subset=["author_id"])
                                 .groupby("author_id")["sentiment"]
                                 .agg(lambda x: x.value_counts().idxmax())
                                 .reset_index()
                                 .rename(columns={"sentiment": "dominant_sentiment"}))
            print(f"[pb-summaries] per-user dominant sentiment cached for {len(_USER_DOM_SENT):,} users; "
                  f"mix = {_USER_DOM_SENT['dominant_sentiment'].value_counts(normalize=True).mul(100).round(1).to_dict()}")
except Exception as _e:
    print(f"[pb-summaries] could not build sentiment cache ({_e}); falling back to random user pick.")

def build_cluster_summary_b(cluster_id, df):
    sub = df[df["macro_cluster"] == cluster_id]
    n = len(sub)
    stat_cols = [c for c in [
        "total_comments", "activity_span_days", "mean_hours_to_comment",
        "pct_comments_under_1h", "reply_ratio", "mean_word_count",
        "emoji_usage_rate", "question_rate", "exclamation_rate",
        # sentiment/toxicity signals — surfaced so the LLM can flag hostile/negative clusters
        "mean_sentiment_score", "pct_negative", "toxicity_rate",
    ] if c in sub.columns]
    mean_stats = sub[stat_cols].mean().round(3).to_dict()

    vibe_mix = {}
    if "dominant_room_vibe" in sub.columns:
        vibe_mix = (sub["dominant_room_vibe"].value_counts(normalize=True)
                    .mul(100).round(1).head(5).to_dict())

    # Pick N_USERS_FOR_LLM representative users, STRATIFIED by each user's dominant sentiment,
    # so every cluster contributes its positive/neutral/negative voices (not 5 random users that
    # would almost always be positive). Include ALL stored comments per picked user.
    N_USERS_FOR_LLM = 12
    sample_users = []
    if "top_comments_sample" in sub.columns:
        picks_pool = sub[["author_id", "top_comments_sample"]].dropna(subset=["top_comments_sample"]).copy()
        # attach per-user dominant sentiment (module-level cache built once: _USER_DOM_SENT)
        if "_USER_DOM_SENT" in globals() and len(picks_pool):
            picks_pool = picks_pool.merge(_USER_DOM_SENT, on="author_id", how="left")
            picks_pool["dominant_sentiment"] = picks_pool["dominant_sentiment"].fillna("neutral")
        else:
            picks_pool["dominant_sentiment"] = "neutral"

        if len(picks_pool):
            # proportional allocation across sentiment strata, but guarantee >=1 from any
            # non-empty minority stratum (esp. negative) when the cluster has room.
            strata = list(picks_pool.groupby("dominant_sentiment"))
            picked_frames = []
            remaining = N_USERS_FOR_LLM
            # first pass: 1 guaranteed seat per non-empty stratum
            for cat, grp in strata:
                if remaining <= 0:
                    break
                picked_frames.append(grp.sample(n=1, random_state=42))
                remaining -= 1
            # second pass: fill the rest proportionally to stratum size
            if remaining > 0:
                already = set(pd.concat(picked_frames)["author_id"]) if picked_frames else set()
                rest_pool = picks_pool[~picks_pool["author_id"].isin(already)]
                if len(rest_pool):
                    take = min(remaining, len(rest_pool))
                    picked_frames.append(
                        rest_pool.groupby("dominant_sentiment", group_keys=False)
                                 .apply(lambda g: g.sample(
                                     n=max(1, round(take * len(g) / len(rest_pool))),
                                     random_state=42))
                                 .head(take))
            picks = pd.concat(picked_frames).drop_duplicates("author_id").head(N_USERS_FOR_LLM)
        else:
            picks = picks_pool

        for _, row in picks.iterrows():
            all_comments = [c.strip() for c in str(row["top_comments_sample"]).split("|||") if c.strip()]
            sample_users.append({
                "user_id": str(row["author_id"]),
                "dominant_sentiment": str(row.get("dominant_sentiment", "neutral")),
                "comments": all_comments,
            })
    # pct_audience is relative to ALL users (incl. noise) so noise share is meaningful.
    return {
        "cluster_id": int(cluster_id),
        "is_noise": cluster_id == -1,
        "n_users": n,
        "pct_audience": round(100 * n / max(len(df), 1), 1),
        "mean_behavioral_stats": mean_stats,
        "dominant_room_vibe_mix": vibe_mix,
        "sample_users": sample_users,
    }

pb_cluster_summaries = [build_cluster_summary_b(c, pb_df) for c in pb_unique_clusters_with_noise]
print(f"Built summaries for {len(pb_cluster_summaries)} clusters (incl. noise):")
for s in pb_cluster_summaries:
    tag = " [NOISE]" if s["is_noise"] else ""
    print(f"   Cluster {s['cluster_id']:>2}{tag}  n={s['n_users']:>6,} ({s['pct_audience']:.1f}%)  "
          f"vibe={list(s['dominant_room_vibe_mix'].items())[:2]}")


[pb-summaries] per-user dominant sentiment cached for 194,012 users; mix = {'positive': 79.8, 'neutral': 15.0, 'negative': 5.1, 'amused': 0.1, 'mixed': 0.0, 'surprise': 0.0, 'question': 0.0, 'joke': 0.0, 'suggestion': 0.0, 'supportive': 0.0, 'anticipation': 0.0, 'amusement': 0.0, 'sarcastic': 0.0, 'affection': 0.0, 'support': 0.0, 'sadness': 0.0}


C:\Users\alire\AppData\Local\Temp\ipykernel_8644\1749417545.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(
C:\Users\alire\AppData\Local\Temp\ipykernel_8644\1749417545.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(
C:\Users\alire\AppData\Local\Temp\ipykernel_8644\1749417545.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. T

Built summaries for 12 clusters (incl. noise):
   Cluster -1 [NOISE]  n= 1,169 (5.8%)  vibe=[('0.4280203045685279', 5.2), ('0.691853023255814', 2.6)]
   Cluster  0  n= 1,039 (5.2%)  vibe=[('0.5120281273131014', 2.1), ('0.4051140833988985', 2.0)]
   Cluster  1  n= 1,587 (7.9%)  vibe=[('0.5753065483955127', 3.1), ('0.6039844329132692', 2.0)]
   Cluster  2  n=   942 (4.7%)  vibe=[('0.4051140833988985', 2.7), ('0.6039844329132692', 2.3)]
   Cluster  3  n=   824 (4.1%)  vibe=[('0.691853023255814', 3.8), ('0.7742907000615891', 3.2)]
   Cluster  4  n= 1,183 (5.9%)  vibe=[('0.4280203045685279', 6.1), ('0.5909688581314879', 5.6)]
   Cluster  5  n=   756 (3.8%)  vibe=[('0.24517184942716858', 5.6), ('0.255562095350206', 2.5)]
   Cluster  6  n=   817 (4.1%)  vibe=[('0.6541747292418773', 8.8), ('0.5753065483955127', 2.4)]
   Cluster  7  n=   893 (4.5%)  vibe=[('0.5909688581314879', 3.4), ('0.24436274509803918', 1.5)]
   Cluster  8  n= 7,120 (35.6%)  vibe=[('0.5753065483955127', 1.1), ('0.6541747292

C:\Users\alire\AppData\Local\Temp\ipykernel_8644\1749417545.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(
C:\Users\alire\AppData\Local\Temp\ipykernel_8644\1749417545.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(
C:\Users\alire\AppData\Local\Temp\ipykernel_8644\1749417545.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. T

In [ ]:
# LLM-label each cluster into a 6-key persona schema (noise included, flagged).
from google.genai import types

_real_summaries  = [s for s in pb_cluster_summaries if not s["is_noise"]]
_noise_summary   = next((s for s in pb_cluster_summaries if s["is_noise"]), None)
# All summaries go to LLM — noise is clearly flagged in the block text.
_all_for_llm     = ([_noise_summary] if _noise_summary else []) + _real_summaries

PB_NAMING_SYSTEM = (
    "You are a strategic audience analyst for an Italian Instagram influencer agency.\n"
    "Commenters were clustered into behavioural macro-segments (UMAP + HDBSCAN). Each cluster is\n"
    "described by mean behavioural stats, a dominant room-vibe mix, and representative comments.\n"
    "\n"
    "NOISE CLUSTER: one cluster is labelled *** NOISE ***. These are HDBSCAN noise points —\n"
    "users too behaviourally scattered to fit any dense cluster. Do NOT simply call them\n"
    "NOISE_UNCLASSIFIED. Instead, examine their stats and comments carefully and give them a\n"
    "descriptive label that captures their dominant observable characteristics (e.g. erratic\n"
    "lurkers, one-off commenters, etc.). Make clear in the description that they lack cluster\n"
    "cohesion, but still characterise what they seem to be doing.\n"
    "\n"
    "LABEL BIAS WARNING: Do NOT default to positive or neutral personas. The dataset may contain\n"
    "critics, trolls, toxic commenters, or passive lurkers. If a cluster's stats show high\n"
    "toxicity_rate, high pct_negative, low word_count, or hostile comment fragments, label it\n"
    "accordingly (e.g. CRITICAL_DETRACTOR, TOXIC_TROLL, PASSIVE_LURKER). Be honest and\n"
    "comprehensive — the full spectrum of audience behaviour must be represented.\n"
    "\n"
    "Turn every cluster into ONE audience persona. Personas must be MUTUALLY EXCLUSIVE.\n"
    f"Return exactly {len(_all_for_llm)} personas ordered by audience share (largest first).\n"
    "For each persona output exactly these 5 keys:\n"
    "  codename (UPPER_SNAKE_CASE),\n"
    "  label (short Title Case, 2-4 words),\n"
    "  description (1-2 sentences — who they are and what drives their engagement),\n"
    "  quantitative_signals (3-5 distinguishing behavioural signals as strings),\n"
    "  example_comments (2-3 verbatim fragments drawn from the cluster's sample comments),\n"
        "Output ONLY a valid JSON array of objects with exactly those 5 keys. No preamble, no markdown fences."
)

def _pb_block(s):
    stats = "  ".join(f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}"
                      for k, v in s["mean_behavioral_stats"].items())
    vibe = "  ".join(f"{k}: {v}%" for k, v in s["dominant_room_vibe_mix"].items())
    noise_tag = "  *** NOISE — low-cohesion, no dominant pattern ***" if s["is_noise"] else ""
    users_block = ""
    for u in s.get("sample_users", []):
        comment_lines = "\n      ".join(f'"{c}"' for c in u["comments"])
        _sent = u.get("dominant_sentiment", "")
        _tag = f" [dominant sentiment: {_sent}]" if _sent else ""
        users_block += f'\n  USER {u["user_id"]}{_tag}:\n      {comment_lines}'
    return (f"CLUSTER {s['cluster_id']}{noise_tag}  (n={s['n_users']:,}, {s['pct_audience']}% of all users)\n"
            f"  Mean behavioural stats: {stats}\n"
            f"  Dominant room-vibe mix: {vibe}\n"
            f"  Representative users (full comment history):{users_block}")

pb_prompt = PB_NAMING_SYSTEM + "\n\n=== CLUSTER DATA ===\n\n" + "\n\n".join(
    _pb_block(s) for s in _all_for_llm)

pb_cfg = types.GenerateContentConfig(
    temperature=0.0, top_p=1.0, max_output_tokens=8192,
    response_mime_type="application/json",
    thinking_config=types.ThinkingConfig(thinking_budget=4096),
)
print(f"Labelling {len(_all_for_llm)} clusters via {MODEL_STAGE3_NAMING} (noise included + flagged) ...")
pb_resp = client.models.generate_content(model=MODEL_STAGE3_NAMING, contents=pb_prompt, config=pb_cfg)

_pb_text = strip_fences(pb_resp.text or "")
try:
    pb_macro_personas = json.loads(_pb_text)
except Exception as e:
    print("Could not parse LLM output:", str(e)[:120], "\n", _pb_text[:400])
    pb_macro_personas = []

_keys = ["codename", "label", "description", "quantitative_signals", "example_comments"]
pb_macro_personas = [{k: p.get(k, "" if k in ("codename","label","description") else [])
                      for k in _keys}
                     for p in pb_macro_personas if isinstance(p, dict)][:TARGET_PERSONAS]

print(f"\n{len(pb_macro_personas)} personas from Pathway B:")
for p in pb_macro_personas:
    is_noise = p["codename"] == "NOISE_UNCLASSIFIED"
    tag = " [NOISE]" if is_noise else ""
    print(f"   {str(p['codename']):<30}{tag}  | {p['label']}")

import os
if os.path.exists(TAXONOMY_JSON_PATH):
    data = json.load(open(TAXONOMY_JSON_PATH, encoding="utf-8"))
else:
    data = {"status": "PENDING_HUMAN_REVIEW", "raw_candidates": []}
data["final_taxonomy"] = pb_macro_personas
data["status"] = "PENDING_HUMAN_REVIEW"
data["pathway"] = "B_CLUSTER"
data["instructions"] = ("Pathway B (UMAP+HDBSCAN) taxonomy. Review/merge, ensure MECE, "
                        "set status='APPROVED' before Stage 2. Set CONSOLIDATION_PATHWAY='B_CLUSTER' "
                        "so the Pathway A wrapper cell does not overwrite this.")
with open(TAXONOMY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print(f"\nWritten -> {TAXONOMY_JSON_PATH} (pathway=B_CLUSTER). "
      "Set CONSOLIDATION_PATHWAY='B_CLUSTER' and status='APPROVED' before Stage 2.")

# cluster_id -> codename map (covers -1 for noise).
_cluster_map = {int(s["cluster_id"]): pb_macro_personas[i]["codename"]
                for i, s in enumerate(pb_cluster_summaries) if i < len(pb_macro_personas)}
with open(CLUSTER_PERSONA_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(_cluster_map, f, ensure_ascii=False, indent=2)
pb_df[["author_id", "macro_cluster"]].to_parquet(
    f"{LOCAL_DIR}/pathway_b_assignments_{PLATFORM}.parquet", index=False)
print(f"Cluster->persona map -> {CLUSTER_PERSONA_MAP_PATH}; "
      f"assignments -> pathway_b_assignments_{PLATFORM}.parquet")


Labelling 12 clusters via gemini-2.5-flash-lite (noise included + flagged) ...

12 personas from Pathway B:
   ENGAGED_FOLLOWERS               | Engaged Followers
   ENTHUSIASTIC_SUPPORTERS         | Enthusiastic Supporters
   EXPRESSIVE_FANS                 | Expressive Fans
   APPRECIATIVE_AUDIENCE           | Appreciative Audience
   ERRATIC_ENGAGERS                | Erratic Engagers
   FRIENDLY_ENGAGERS               | Friendly Engagers
   MINIMAL_COMMENTERS              | Minimal Commenters
   CURIOUS_INQUIRERS               | Curious Inquirers
   EXCITED_ENTHUSIASTS             | Excited Enthusiasts
   QUICK_REACTORS                  | Quick Reactors
   SHORT_SWEET_EMOJIS              | Short & Sweet
   DISGRUNTLED_CRITICS             | Disgruntled Critics

Written -> outputs/taxonomy.json (pathway=B_CLUSTER). Set CONSOLIDATION_PATHWAY='B_CLUSTER' and status='APPROVED' before Stage 2.
Cluster->persona map -> outputs/cluster_persona_map_instagram.json; assignments -> pathway_b_ass

In [ ]:
# Display full persona descriptions
import json, os
from IPython.display import Markdown, display

_taxonomy = (
    _taxonomy if "_taxonomy" in dir()
    else json.load(open(TAXONOMY_JSON_PATH, encoding="utf-8")).get("final_taxonomy", [])
)

for persona in _taxonomy:
    codename = persona["codename"]
    label = persona["label"]
    description = persona["description"]
    signals = persona.get("quantitative_signals", [])
    examples = persona.get("example_comments", [])

    signals_md = "\n".join(f"- {s}" for s in signals)
    examples_md = "\n".join(f"> {e}" for e in examples)

    md = f"""
### **{codename}** — {label}

{description}

**Key signals:**  
{signals_md}

**Example comments:**  
{examples_md}

---
"""
    display(Markdown(md))



### **LOYAL_SUSTAINERS** — Loyal Followers

These are the most dedicated followers, exhibiting long-term engagement and frequent commenting. They often use emojis and engage in personal conversations or plans.

**Key signals:**  
- total_comments=4.88
- activity_span_days=778.92
- emoji_usage_rate=0.64
- question_rate=0.09

**Example comments:**  
> A vita alta ora è per sempre! 🙌
> ho bisogno di parlarti urgentemente possiamo sentirci in privato grazie
> Io aspetto il matrimonio 🤷🏻‍♀️😂buon compleanno 🎉

---



### **ENTHUSIASTIC_EMOTICONS** — Enthusiastic Emojis

These users express strong positive sentiment through very frequent emoji use and short, punchy comments. They are highly appreciative but offer little detailed feedback.

**Key signals:**  
- total_comments=1.18
- mean_word_count=5.78
- emoji_usage_rate=1.00
- question_rate=0.00

**Example comments:**  
> Sei sempre una bomba 💣❤️
> Tiamo 🥇
> 🤣🤣🤣 ... Non male 3 secondi 🤣

---



### **EMOJI_EXPRESSIVES** — Emoji Expressives

These users communicate primarily through a high volume of emojis, often in short, positive comments. They express enthusiasm and agreement with minimal text.

**Key signals:**  
- total_comments=1.07
- mean_word_count=5.08
- emoji_usage_rate=1.00
- exclamation_rate=0.00

**Example comments:**  
> @ilariarisolino finalmentee😍
> @camihawke 🤣🤣🤣🤣🤣🤣🤣🤣🤣🤣🤣🤣🤣🤣
> @ilydef 😍😍😍😍

---



### **OBSERVANT_WITTERS** — Observant Witters

These users engage with brief, often humorous or observational comments, showing little emotional expression through emojis or exclamations. They comment sporadically but can be witty.

**Key signals:**  
- total_comments=1.09
- mean_word_count=6.96
- emoji_usage_rate=0.00
- exclamation_rate=0.00

**Example comments:**  
> Ahahahahaahah
> Torino ha letteralmente sbagliato paese
> non ho mai fatto un ordine tanto in fretta in tutta la mia vita

---



### **ERRATIC_REACTORS** — Erratic Reactors

These users are behaviourally scattered and comment only once, lacking consistent engagement patterns. They often use exclamations, suggesting impulsive reactions to content without deeper interaction.

**Key signals:**  
- total_comments=1.07
- activity_span_days=10.69
- exclamation_rate=0.41
- mean_word_count=7.84

**Example comments:**  
> Il metodo Kominski
> Verissimooooooo!!
> Le 4 stagioni!!!! E noncielodiconoooooo!!

---


...


### **THOUGHTFUL_DISCUSSANTS** — Thoughtful Discussants

These users leave longer, more detailed comments, often asking questions or sharing advice and personal experiences. They engage thoughtfully with the content.

**Key signals:**  
- total_comments=1.05
- mean_word_count=24.76
- emoji_usage_rate=0.44
- question_rate=0.12

**Example comments:**  
> @camillagrignani voglio quelle gambe 😭😭😭
> ALL'OVERTHINKER: essere curiosə, se ha un dubbio chieda, se inizia a rimuginare chieda, non farsi travolgere dall'onda ma imparare a surfarci. E comunque se chi è dalla parte opposta non è propenso a determinate dinamiche non è colpa tua, l'altra persona è cosi e poco ci si puo fare.
Ciau
> fai un trito di polpette di lenticchie in modo da usarlo come se fosse della carne macinata per il ragù. usa il tutto per fare la tua pasta pasticciata più buona dell'universo... e stappa un buon vino che aiuta a buttar giù il tutto:)

---



### **QUICK_REACTORS** — Quick Reactors

These users engage with very short, emoji-heavy comments, often expressing strong positive sentiment or agreement. They react quickly and enthusiastically.

**Key signals:**  
- total_comments=1.03
- mean_word_count=2.07
- emoji_usage_rate=1.00
- exclamation_rate=0.00

**Example comments:**  
> ❤️❤️❤️
> 💪🏻
> 😍😍😍

---



### **INQUISITIVE_ENGAGERS** — Inquisitive Engagers

These users tend to ask questions and engage more deeply with content, seeking information or clarification. They show moderate word count and emoji usage.

**Key signals:**  
- total_comments=1.04
- mean_word_count=10.66
- emoji_usage_rate=0.46
- question_rate=1.00

**Example comments:**  
> Sono d'accordo....ma più che altro perché io mi chiamo Giancarlo come lui?!😒
> @camihawke alla fine dove avevi mangiato?!?non ci sono le story in evidenza mannaggia a me che non me lo sono segnata!
> Ciao!Ma questo posto come si chiama?

---



### **MINIMALIST_TAGGERS** — Minimalist Taggers

These users engage with extremely brief comments, often just single words or tags, showing minimal activity and depth. They are present but contribute very little substance.

**Key signals:**  
- total_comments=1.05
- activity_span_days=7.12
- mean_word_count=2.62
- emoji_usage_rate=0.00

**Example comments:**  
> @instaqueen20_10
> @giorgiajoarg totale
> @effetticollaterali_ si vabbe

---



### **CASUAL_OBSERVERS** — Casual Observers

These users comment infrequently with very short, emoji-heavy messages, often expressing simple appreciation or agreement. They are passive but positive.

**Key signals:**  
- total_comments=1.02
- mean_word_count=0.66
- emoji_usage_rate=1.00
- exclamation_rate=0.00

**Example comments:**  
> 😍😍
> 🤣🤣🤣
> 💕

---


## Persona × Sentiment & Toxicity Analysis

These charts show how **sentiment** and **toxicity** distribute across personas:

- **Sentiment mix**: what fraction of each persona's users are negative/neutral/positive
- **Where do negative users land**: which personas capture the most negative-dominant users (% of all negatives)
- **Toxicity mix**: internal toxicity breakdown within each persona
- **Where do toxic users land**: which personas capture the most toxic users (% of all toxic users)
- **Toxic vs non-toxic split**: proportion of toxic users within each persona

NOISE_UNCLASSIFIED users are excluded from all charts.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import json, os

# ── load data ──────────────────────────────────────────────────────────────
_pb = pd.read_parquet(f"{LOCAL_DIR}/pathway_b_assignments_{PLATFORM}.parquet")
_pb["author_id"] = _pb["author_id"].astype(str)

_sent_raw = pd.read_parquet(SENTIMENT_PATH)
if "author_id" not in _sent_raw.columns:
    raise ValueError("sentiment parquet lacks author_id — join via comment_id first")
_sent_raw["author_id"] = _sent_raw["author_id"].astype(str)
if "sentiment" in _sent_raw.columns:
    _sent_raw["sentiment"] = canon_sentiment(_sent_raw["sentiment"])

# dominant sentiment per user
_dom_sent = (_sent_raw.groupby("author_id")["sentiment"]
              .agg(lambda x: x.value_counts().idxmax())
              .reset_index()
              .rename(columns={"sentiment": "dominant_sentiment"}))

# dominant toxicity per user
_has_tox = "toxicity" in _sent_raw.columns
if _has_tox:
    _dom_tox = (_sent_raw.groupby("author_id")["toxicity"]
                .agg(lambda x: x.value_counts().idxmax())
                .reset_index()
                .rename(columns={"toxicity": "dominant_toxicity"}))
    # is_toxic_user: at least one comment flagged mild or severe
    _user_any_tox = (_sent_raw.assign(
        _tox_flag=_sent_raw["toxicity"].isin(["mild", "severe"])
    ).groupby("author_id")["_tox_flag"].any()
     .reset_index().rename(columns={"_tox_flag": "is_toxic_user"}))

_cmap = json.load(open(CLUSTER_PERSONA_MAP_PATH, encoding="utf-8"))
_cmap_int = {int(k): v for k, v in _cmap.items()}

_df = _pb.merge(_dom_sent, on="author_id", how="left")
if _has_tox:
    _df = _df.merge(_dom_tox, on="author_id", how="left")
    _df = _df.merge(_user_any_tox, on="author_id", how="left")
    _df["is_toxic_user"] = _df["is_toxic_user"].fillna(False)
else:
    _df["dominant_toxicity"] = "unknown"
    _df["is_toxic_user"] = False

_df["persona"] = _df["macro_cluster"].map(_cmap_int).fillna("UNKNOWN")
# keep NOISE in df but label it; exclude from per-persona breakdowns below
_df_all  = _df.copy()
_df_real = _df[_df["macro_cluster"] != -1].copy()

os.makedirs(f"{LOCAL_DIR}/figs", exist_ok=True)

SENT_ORDER  = ["negative", "neutral", "positive"]
SENT_COLORS = {"negative": "#d62728", "neutral": "#aec7e8", "positive": "#2ca02c"}
TOX_ORDER   = ["none", "mild", "severe", "spam_promo"]
TOX_COLORS  = {"none": "#9ecae1", "mild": "#fdae6b", "severe": "#d62728", "spam_promo": "#756bb1"}

# ── persona descriptions ──────────────────────────────────────────────────────
_tax_data = json.load(open(TAXONOMY_JSON_PATH, encoding="utf-8"))
_tax_list = _tax_data.get("final_taxonomy", [])
_tax_by_code = {p["codename"]: p for p in _tax_list}
print("=" * 60)
print("PERSONA DESCRIPTIONS")
print("=" * 60)
for _persona_name in _pct_sent.index:
    _p = _tax_by_code.get(_persona_name, {})
    _lbl = _p.get("label", "")
    _desc = _p.get("description", "No description available.")
    print(f"\n{_persona_name}" + (f" — {_lbl}" if _lbl else ""))
    print(f"  {_desc}")
print("\n" + "=" * 60 + "\n")

# ── figure 1: stacked 100% bar — sentiment mix per persona ────────────────
_ct_sent = (_df_real.dropna(subset=["dominant_sentiment"])
            .groupby(["persona", "dominant_sentiment"])
            .size().unstack(fill_value=0)
            .reindex(columns=[c for c in SENT_ORDER
                              if c in _df_real["dominant_sentiment"].dropna().unique()],
                     fill_value=0))
_pct_sent = _ct_sent.div(_ct_sent.sum(axis=1), axis=0) * 100
_pct_sent = _pct_sent.loc[_ct_sent.sum(axis=1).sort_values(ascending=False).index]

fig1, ax1 = plt.subplots(figsize=(11, 5))
bottom = pd.Series(0.0, index=_pct_sent.index)
for sent in [c for c in SENT_ORDER if c in _pct_sent.columns]:
    ax1.bar(_pct_sent.index, _pct_sent[sent], bottom=bottom,
            label=sent.capitalize(), color=SENT_COLORS[sent], alpha=0.88)
    bottom += _pct_sent[sent]
ax1.set_title("Sentiment Mix per Persona (% of each persona's users)", fontsize=13, pad=10)
ax1.set_xlabel("Persona (sorted by size)")
ax1.set_ylabel("% of users in persona")
ax1.set_ylim(0, 100)
ax1.legend(title="Dominant sentiment", loc="upper right", fontsize=9)
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig(f"{LOCAL_DIR}/figs/persona_sentiment_mix.png", dpi=150)
plt.show()
print("saved persona_sentiment_mix.png")

# ── figure 2: where do ALL negative users land? (share of total negatives) ─
_neg_users = _df_real[_df_real["dominant_sentiment"] == "negative"]
_total_neg = max(len(_neg_users), 1)
_neg_dist  = _neg_users.groupby("persona").size().sort_values(ascending=False)
_neg_pct   = (_neg_dist / _total_neg * 100)

fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.bar(_neg_pct.index, _neg_pct.values, color=SENT_COLORS["negative"], alpha=0.85)
ax2.set_title(
    f"Where Do Negative Users Land? (share of all {_total_neg:,} negative-dominant users)",
    fontsize=12, pad=10)
ax2.set_xlabel("Persona")
ax2.set_ylabel("% of ALL negative users")
ax2.yaxis.set_major_locator(mticker.MaxNLocator(integer=False))
for bar, pct in zip(ax2.patches, _neg_pct.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{pct:.1f}%", ha="center", va="bottom", fontsize=8)
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig(f"{LOCAL_DIR}/figs/persona_negative_dist.png", dpi=150)
plt.show()
print("saved persona_negative_dist.png")

# ── figures 3-5: toxicity charts ──────────────────────────────────────────
if _has_tox:
   

    # figure 4: where do ALL toxic users land? (share of total toxic users per persona)
    _tox_users  = _df_real[_df_real["is_toxic_user"] == True]
    _total_tox  = max(len(_tox_users), 1)
    _tox_dist   = _tox_users.groupby("persona").size().sort_values(ascending=False)
    _tox_pct    = _tox_dist / _total_tox * 100

    fig4, ax4 = plt.subplots(figsize=(10, 5))
    bars = ax4.bar(_tox_pct.index, _tox_pct.values,
                   color="#d62728", alpha=0.82)
    ax4.set_title(
        f"Where Do Toxic Users Land? (share of all {_total_tox:,} toxic users)",
        fontsize=12, pad=10)
    ax4.set_xlabel("Persona")
    ax4.set_ylabel("% of ALL toxic users")
    for bar, pct in zip(bars, _tox_pct.values):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{pct:.1f}%", ha="center", va="bottom", fontsize=8)
    plt.xticks(rotation=35, ha="right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{LOCAL_DIR}/figs/persona_toxic_dist.png", dpi=150)
    plt.show()
    print("saved persona_toxic_dist.png")

    # figure 5: toxic vs non-toxic users — proportional breakdown per persona
    _tox_ct = (_df_real.groupby("persona")["is_toxic_user"]
               .value_counts().unstack(fill_value=0)
               .rename(columns={True: "toxic", False: "non-toxic"})
               .reindex(columns=["non-toxic", "toxic"], fill_value=0))
    _tox_ct_pct = _tox_ct.div(_tox_ct.sum(axis=1), axis=0) * 100
    _tox_ct_pct = _tox_ct_pct.loc[_tox_ct.sum(axis=1).sort_values(ascending=False).index]

    fig5, ax5 = plt.subplots(figsize=(11, 4))
    ax5.bar(_tox_ct_pct.index, _tox_ct_pct.get("non-toxic", 0),
            label="Non-toxic", color="#9ecae1", alpha=0.88)
    ax5.bar(_tox_ct_pct.index, _tox_ct_pct.get("toxic", 0),
            bottom=_tox_ct_pct.get("non-toxic", 0),
            label="Toxic (mild+severe)", color="#d62728", alpha=0.88)
    ax5.set_title("Toxic vs Non-Toxic User Split per Persona (%)", fontsize=13, pad=10)
    ax5.set_xlabel("Persona (sorted by size)")
    ax5.set_ylabel("% of users in persona")
    ax5.set_ylim(0, 100)
    ax5.legend(loc="upper right", fontsize=9)
    plt.xticks(rotation=35, ha="right", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{LOCAL_DIR}/figs/persona_toxic_split.png", dpi=150)
    plt.show()
    print("saved persona_toxic_split.png")
else:
    print("No toxicity column found — skipping toxicity charts.")

# ── summary table ──────────────────────────────────────────────────────────
print(f"\nPersona  |  total  |  neg%  |  pos%  |  toxic% (of persona)  |  toxic% (of all toxic)")
print("-" * 80)
_total_tox_all = max(int(_df_real["is_toxic_user"].sum()), 1) if _has_tox else 1
for p in _pct_sent.index:
    total = int(_ct_sent.loc[p].sum())
    neg_pct = _pct_sent.loc[p].get("negative", 0)
    pos_pct = _pct_sent.loc[p].get("positive", 0)
    if _has_tox and p in _tox_ct.index:
        tox_internal = float(_tox_ct.loc[p].get("toxic", 0) / max(_tox_ct.loc[p].sum(), 1) * 100)
        tox_share    = float(_tox_ct.loc[p].get("toxic", 0) / _total_tox_all * 100)
    else:
        tox_internal = tox_share = 0.0
    print(f"  {p:<32s} {total:5d}  {neg_pct:5.1f}%  {pos_pct:5.1f}%  "
          f"{tox_internal:6.1f}%                {tox_share:6.1f}%")


## Stage 1.5 — Auto-Consolidate Candidates → Draft Taxonomy

Collapses the raw discovery candidates' naming variants (e.g. `TAGGER` / `THE TAGGER` / `THE_TAGGER`)
into themes, ranks them by how many candidates fall in each, and drafts the top-`k` as a starter
`final_taxonomy` — merged description + signals + example comments per persona.

Still a **draft**: string-normalisation can't merge *semantically* similar themes
(`TAGGER`/`CONNECTOR`, `ADMIRER`/`APPRECIATOR`/`CHEERLEADER`), so review, merge those by hand, trim to
a clean MECE set, drop `_n_candidates`, and set `status="APPROVED"` before Stage 2.

In [123]:
import json, re
from collections import Counter, defaultdict

def draft_taxonomy_from_candidates(path=TAXONOMY_JSON_PATH, top_k=12, min_count=1, write=True):
    """Group raw candidates by naming-normalised codename, rank by frequency, and draft
    the top_k as a starter final_taxonomy (does NOT approve — you review + set status)."""
    data = json.load(open(path, encoding="utf-8"))
    raw = [c for c in data.get("raw_candidates", []) if isinstance(c, dict) and c.get("codename")]
    if not raw:
        print("No raw_candidates — run/retrieve Stage 1 first.")
        return []

    def norm(c):
        c = re.sub(r"[^A-Z0-9 ]", " ", c.upper().replace("_", " "))
        c = re.sub(r"\s+", " ", c).strip()
        return c[4:].strip() if c.startswith("THE ") else c

    groups = defaultdict(list)
    for c in raw:
        groups[norm(c["codename"])].append(c)
    ranked = [(k, v) for k, v in sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True)
              if len(v) >= min_count][:top_k]

    def merge_list(cands, key, limit=6):
        seen, out = set(), []
        for c in cands:
            for item in (c.get(key) or []):
                s = str(item).strip()
                if s and s.lower() not in seen:
                    seen.add(s.lower()); out.append(s)
        return out[:limit]

    draft = []
    for theme, cands in ranked:
        canon = Counter(c["codename"].strip().upper().replace(" ", "_")
                        for c in cands).most_common(1)[0][0]
        draft.append({
            "codename": canon,
            "label": theme.title(),
            "description": max((c.get("description", "") for c in cands), key=len, default=""),
            "quantitative_signals": merge_list(cands, "signals"),
            "example_comments": merge_list(cands, "examples"),
            "_n_candidates": len(cands),   # how many raw candidates merged — your cue for importance; remove before approving
        })

    covered = sum(d["_n_candidates"] for d in draft)
    print(f"{len(raw)} candidates -> {len(groups)} themes. Drafted top {len(draft)} "
          f"(cover {covered}/{len(raw)} = {100*covered/len(raw):.0f}% of candidates):")
    for d in draft:
        print(f"  {d['codename']:<28} merged {d['_n_candidates']:>3} | {d['label']}")

    if write:
        data["final_taxonomy"] = draft
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"\n✏️  Draft written to {path} -> final_taxonomy.")
        print("    Next: MERGE semantically-similar personas, trim to a MECE set,")
        print("    remove '_n_candidates', then set status='APPROVED'.")
    return draft

# Tune top_k / min_count to taste; re-run freely (it only rewrites final_taxonomy).
_draft = draft_taxonomy_from_candidates(top_k=15)

19718 candidates -> 2870 themes. Drafted top 15 (cover 5062/19718 = 26% of candidates):
  THE_ADMIRER                  merged 697 | Admirer
  ENGAGED_FAN                  merged 567 | Engaged Fan
  THE_TAGGER                   merged 433 | Tagger
  SOCIAL_CONNECTOR             merged 418 | Social Connector
  THE_ENTHUSIAST               merged 378 | Enthusiast
  CASUAL_OBSERVER              merged 331 | Casual Observer
  THE_APPRECIATOR              merged 328 | Appreciator
  SUPERFAN                     merged 279 | Superfan
  SOCIAL_SHARER                merged 266 | Social Sharer
  EMOJI_APPRECIATOR            merged 245 | Emoji Appreciator
  CASUAL_APPRECIATOR           merged 243 | Casual Appreciator
  THE_CHEERLEADER              merged 235 | Cheerleader
  EMOJI_EXPRESSER              merged 222 | Emoji Expresser
  HUMOR_APPRECIATOR            merged 210 | Humor Appreciator
  THE_LAUGHER                  merged 210 | Laugher

✏️  Draft written to outputs/taxonomy.json -> final_ta

### Alternative — LLM Consolidation (one Gemini call, higher quality)

Instead of the string-normalised draft above, this hands ALL raw candidates to Gemini Pro and asks
for a single clean **MECE** taxonomy of ≤ `target_personas`, merging *semantic* synonyms
(`TAGGER`/`CONNECTOR`, `ADMIRER`/`APPRECIATOR`/`CHEERLEADER`) that string-matching can't. One online
call (not batch). Writes the result to `final_taxonomy`; you still review and set `status="APPROVED"`.
The two cells are alternatives — run whichever you prefer (each overwrites `final_taxonomy`).

In [66]:
import json, re
from collections import defaultdict, Counter
from google.genai import types

def _norm_codename(c):
    c = re.sub(r"[^A-Z0-9 ]", " ", c.upper().replace("_", " "))
    c = re.sub(r"\s+", " ", c).strip()
    return c[4:].strip() if c.startswith("THE ") else c

def _merge_unique(cands, key, limit):
    seen, out = set(), []
    for c in cands:
        for item in (c.get(key) or []):
            s = str(item).strip()
            if s and s.lower() not in seen:
                seen.add(s.lower()); out.append(s)
    return out[:limit]

def llm_consolidate_taxonomy(path=TAXONOMY_JSON_PATH, target_personas=TARGET_PERSONAS,
                             model=MODEL_STAGE2_CLASSIFY, max_themes=400,
                             max_prompt_tokens=MAX_BATCH_TOKENS, write=True):
    """Two-stage consolidation that stays under the model token limit.

    1. Deterministically collapse the (up to ~20k) raw candidates into naming-normalised
       THEMES, ranked by how many candidates fall in each.
    2. Feed only the top `max_themes` themes (merged signals/examples, NOT every raw
       candidate) to Gemini for a final MECE taxonomy. This is ~400 compact records
       instead of ~20k, so it fits comfortably under the 1M token limit.
    """
    data = json.load(open(path, encoding="utf-8"))
    raw = [c for c in data.get("raw_candidates", []) if isinstance(c, dict) and c.get("codename")]
    if not raw:
        print("No raw_candidates — run/retrieve Stage 1 first.")
        return []

    # ── stage 1: deterministic theme grouping ────────────────────────────────
    groups = defaultdict(list)
    for c in raw:
        groups[_norm_codename(c["codename"])].append(c)
    ranked = sorted(groups.items(), key=lambda kv: len(kv[1]), reverse=True)

    themes = []
    for theme, cands in ranked[:max_themes]:
        canon = Counter(c["codename"].strip().upper().replace(" ", "_")
                        for c in cands).most_common(1)[0][0]
        themes.append({
            "codename": canon,
            "n_candidates": len(cands),
            "description": max((c.get("description", "") for c in cands), key=len, default="")[:220],
            "signals": _merge_unique(cands, "signals", 4),
            "examples": _merge_unique(cands, "examples", 3),
        })
    covered = sum(t["n_candidates"] for t in themes)
    print(f"Stage 1.5: {len(raw):,} candidates -> {len(groups):,} themes; "
          f"sending top {len(themes)} to LLM (cover {100*covered/len(raw):.0f}% of candidates).")

    # ── trim to fit token budget (4 chars ~= 1 token) ────────────────────────
    def _fits(lst):
        return len(json.dumps(lst, ensure_ascii=False)) // 4 < max_prompt_tokens
    while themes and not _fits(themes):
        themes = themes[: int(len(themes) * 0.8)]
    print(f"   {len(themes)} themes after token-budget trim (~{len(json.dumps(themes, ensure_ascii=False))//4:,} tokens).")

    system = (
        f"You are consolidating {len(themes)} candidate audience-persona THEMES — auto-grouped from an "
        "Italian Instagram influencer's comment community — into ONE clean, MECE taxonomy.\n"
        "Each theme carries n_candidates (how many raw candidates it represents — a popularity signal).\n"
        f"Merge overlapping themes into AT MOST {target_personas} distinct, non-overlapping personas that "
        "together cover the audience. Order them by importance (use n_candidates as a guide).\n"
        "\nLABEL BIAS WARNING: Do NOT collapse everything into positive/fan personas. Italian comment "
        "communities contain critics, detractors, sarcastic/hostile voices, taggers, and passive lurkers. "
        "If themes describe negativity, criticism of the influencer (incl. appearance), sarcasm, or "
        "hostility, PRESERVE them as their own persona (e.g. CRITICAL_DETRACTOR, SARCASTIC_SKEPTIC). "
        "A taxonomy that is 100% positive is wrong.\n"
        "\nFor each final persona output exactly these 5 keys: codename (UPPER_SNAKE_CASE), "
        "label (short Title Case), description (1-2 sentences), quantitative_signals (3-5 strings), "
        "example_comments (2-3 verbatim fragments drawn from the themes).\n"
        "Output ONLY a valid JSON array of objects with exactly those 5 keys. No preamble, no markdown fences."
    )
    prompt = system + "\n\n=== THEMES ===\n" + json.dumps(themes, ensure_ascii=False)

    cfg = types.GenerateContentConfig(
        temperature=0.0, top_p=1.0, max_output_tokens=8192,
        response_mime_type="application/json",
        thinking_config=types.ThinkingConfig(thinking_budget=4096),
    )
    print(f"Consolidating via {model} ...")
    resp = client.models.generate_content(model=model, contents=prompt, config=cfg)

    text = strip_fences(resp.text or "")
    try:
        final = json.loads(text)
    except Exception as e:
        print("Could not parse LLM output:", str(e)[:120], "\n", text[:400])
        return []
    if not isinstance(final, list):
        print("Expected a JSON array, got", type(final).__name__)
        return []

    keys = ["codename", "label", "description", "quantitative_signals", "example_comments"]
    final = [{k: p.get(k, "" if k in ("codename", "label", "description") else []) for k in keys}
             for p in final if isinstance(p, dict)]

    print(f"\n{len(final)} consolidated personas:")
    for p in final:
        print(f"  {str(p['codename']):<28} | {p['label']}")
    if write:
        data["final_taxonomy"] = final
        data["pathway"] = "A_LLM"
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"\nWritten to {path} -> final_taxonomy. Review/tweak, then set status='APPROVED'.")
    return final

# Uncomment to run (one online Gemini call):
_final = llm_consolidate_taxonomy(target_personas=TARGET_PERSONAS)


Consolidating 19718 candidates -> <= 12 personas via gemini-2.5-pro …


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'The input token count (1964479) exceeds the maximum number of tokens allowed (1048576).', 'status': 'INVALID_ARGUMENT'}}

In [43]:
import json

if CONSOLIDATION_PATHWAY == "A_LLM" and "_final" in dir():
    taxonomy_wrapper = {
        "status": "PENDING_HUMAN_REVIEW",
        "pathway": "A_LLM",
        "instructions": "Consolidate the candidates into a MECE taxonomy. Each final persona needs a unique 'codename' (plus 'label', 'description', 'quantitative_signals', 'example_comments'). Set status='APPROVED' before Stage 2.",
        "final_taxonomy": _final
    }
    with open(f"{LOCAL_DIR}/taxonomy.json", 'w', encoding="utf-8") as f:
        json.dump(taxonomy_wrapper, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved {len(_final)} personas to {LOCAL_DIR}/taxonomy.json")
else:
    print("Skipping Pathway A wrapper (Pathway B taxonomy already written, or _final absent).")


✅ Saved 0 personas to outputs/taxonomy.json


## Stage 2 — Deterministic Classification (Batch · Multimodal)

**One user per JSONL line.** Each request carries the approved taxonomy (system text), the user's
behavioural profile, and their engaged-post media. Pro returns exactly one persona per user with a
confidence and a cited justification; we join back on the `author_id` the model echoes.

In [44]:
def build_stage2_system_prompt(taxonomy: list) -> str:
    taxonomy_text = ""
    for p in taxonomy:
        taxonomy_text += (
            f"\nPERSONA: {p['codename']} — {p.get('label', '')}\n"
            f"  Description: {p.get('description', '')}\n"
            f"  Quantitative signals: {'; '.join(p.get('quantitative_signals', []))}\n"
            f"  Example comments: {' | '.join(p.get('example_comments', []))}\n"
        )
    return (
        "You are a deterministic community analyst for Show Reel Media Group.\n"
        "Classify the Instagram commenter into exactly ONE persona from the approved taxonomy,\n"
        "using their behavioural metrics, comment samples, AND the attached post media (images /\n"
        "video frames + transcript) of the content they engage with most.\n\n"
        "=== APPROVED PERSONA TAXONOMY ===\n"
        f"{taxonomy_text}"
        "=================================\n\n"
        "RULES:\n"
        "1. Assign exactly ONE persona — the closest match.\n"
        "2. Output a confidence score between 0.0 and 1.0.\n"
        "3. Cite a specific comment fragment or media detail as justification.\n"
        "4. If data is insufficient, assign the most probable persona with confidence <= 0.4.\n"
        "5. Echo the author_id exactly as given.\n"
        "6. Output ONLY a single valid JSON object. No preamble, no markdown fences.\n\n"
        'Schema: {"author_id": str, "persona_codename": str, "confidence": float, "justification": str}'
    )

def format_user_profile_for_stage2(row) -> dict:
    return {
        "author_id":                str(row["author_id"]),
        "total_comments":           int(row["total_comments"]),
        "unique_posts":             int(row["unique_posts_commented"]),
        "activity_span_days":       int(row["activity_span_days"]),
        "pct_comments_under_1h":    round(float(row["pct_comments_under_1h"]), 2),
        "pct_comments_under_24h":   round(float(row["pct_comments_under_24h"]), 2),
        "reply_ratio":              round(float(row["reply_ratio"]), 2),
        "mean_mention_count":       round(float(row["mean_mention_count"]), 2),
        "mean_word_count":          round(float(row["mean_word_count"]), 1),
        "emoji_usage_rate":         round(float(row["emoji_usage_rate"]), 2),
        "question_rate":            round(float(row["question_rate"]), 2),
        "exclamation_rate":         round(float(row["exclamation_rate"]), 2),
        "post_concentration_ratio": round(float(row["post_concentration_ratio"]), 2),
        "sample_comments":          str(row.get("top_comments_sample", ""))[:500],
    }

def build_stage2_line(row, system_prompt: str) -> dict:
    profile = json.dumps(format_user_profile_for_stage2(row), ensure_ascii=False)
    parts = [{"text": system_prompt + "\n\n=== USER TO CLASSIFY ===\n" + profile +
              "\n\nThe images/frames and transcripts below are sample posts this user engaged with most."}]
    parts.extend(build_user_media_parts(row["author_id"]))
    parts.append({"text": "\nClassify THIS user into exactly one persona. Output ONLY one JSON object."})
    return {"request": {"contents": [{"role": "user", "parts": parts}],
                        "generationConfig": gen_config_dict(MAX_OUTPUT_TOKENS_STAGE2, THINKING_BUDGET_STAGE2)}}

# ── SUBMIT (returns immediately — safe to close the laptop) ────────────────────
def submit_stage2(user_features_df, taxonomy, max_users=STAGE2_MAX_USERS):
    df = user_features_df if max_users is None else user_features_df.head(max_users)
    print(f"\n{'='*60}\nSTAGE 2 (batch submit) — Classification\n"
          f"  Users: {len(df):,} (1/request) | Model: {MODEL_STAGE2_CLASSIFY}\n{'='*60}")
    system_prompt = build_stage2_system_prompt(taxonomy)
    lines = [build_stage2_line(row, system_prompt)
             for _, row in tqdm(df.iterrows(), total=len(df), desc="Build Stage 2 requests")]
    in_uri  = upload_to_gcs(write_jsonl(lines, f"{LOCAL_DIR}/stage2_input.jsonl"),
                            GCS_BUCKET, BATCH_INPUT_PREFIX + "stage2_input.jsonl")
    out_uri = f"gs://{GCS_BUCKET}/{BATCH_OUTPUT_PREFIX}stage2/"
    job = submit_batch_job(in_uri, out_uri, MODEL_STAGE2_CLASSIFY)
    record_batch_job("stage2", job, out_uri)
    print(f"\n\U0001f4e4 Stage 2 submitted ({len(lines)} requests). Safe to close the laptop.")
    print("   When it finishes, run:  retrieve_stage2()   -> writes user_personas.parquet")
    return job

# ── RETRIEVE (run later; rebuilds the per-user join from user_features) ────────
def retrieve_stage2(user_features_df=None, max_users=STAGE2_MAX_USERS, output_path=RESULTS_PATH):
    job = get_recorded_job("stage2")
    if job.state != JobState.JOB_STATE_SUCCEEDED:
        print(f"⏳ Stage 2 not ready (state={job.state}). Re-run later.")
        return None
    if user_features_df is None:
        user_features_df = user_features
    df = user_features_df if max_users is None else user_features_df.head(max_users)

    results = []
    for t in retrieve_response_texts(job, GCS_BUCKET):
        if not t:
            continue
        try:
            obj = json.loads(strip_fences(t))
            if isinstance(obj, list):
                results.extend(obj)
            elif isinstance(obj, dict):
                results.append(obj)
        except Exception as e:
            print("   parse error:", str(e)[:80])

    results_df = pd.DataFrame(results)
    if "author_id" in results_df.columns:
        results_df["author_id"] = results_df["author_id"].astype(str)
    summary_cols = ["author_id", "total_comments", "activity_span_days",
                    "mean_hours_to_comment", "pct_comments_under_1h",
                    "reply_ratio", "mean_word_count"]
    base = df.copy()
    base["author_id"] = base["author_id"].astype(str)
    final_df = base[summary_cols].merge(results_df, on="author_id", how="left")
    final_df.to_parquet(output_path, index=False)

    matched = final_df["persona_codename"].notna().sum() if "persona_codename" in final_df.columns else 0
    print(f"\n✅ Stage 2 complete. Classified {matched:,}/{len(final_df):,} users -> {output_path}")
    if "persona_codename" in final_df.columns:
        dist = final_df["persona_codename"].value_counts(normalize=True).mul(100).round(1)
        print("\n   Persona Distribution:")
        for persona, pct in dist.items():
            print(f"      {str(persona):<35} {pct:.1f}%")
    return final_df

print("✅ Stage 2 functions ready (submit_stage2 / retrieve_stage2).")

✅ Stage 2 functions ready (submit_stage2 / retrieve_stage2).


## Persona Map Export (author_id -> persona, joinable to ig_comments_clean)

A clean `author_id -> persona_codename` map so the assigned persona can be attached as a
feature to the cleaned comments table later. Works identically for both pathways because
it reads the Stage 2 output (`user_personas.parquet`), which is pathway-agnostic.

In [247]:
def export_persona_map(results_path=RESULTS_PATH, out_path=PERSONA_MAP_PATH):
    """author_id -> persona_codename(, confidence), one row per author, for downstream joins."""
    import os, pandas as pd
    if not os.path.exists(results_path):
        raise FileNotFoundError(f"{results_path} missing — run retrieve_stage2() first.")
    df = pd.read_parquet(results_path)
    df["author_id"] = df["author_id"].astype(str)
    cols = ["author_id", "persona_codename"] + (["confidence"] if "confidence" in df.columns else [])
    pmap = (df[cols].dropna(subset=["persona_codename"])
                    .drop_duplicates("author_id").reset_index(drop=True))
    pmap.to_parquet(out_path, index=False)
    print(f"✅ Persona map: {len(pmap):,} authors -> {out_path}")
    print("   Attach downstream via:")
    print("     ig_comments_clean = ig_comments_clean.merge("
          "pd.read_parquet(PERSONA_MAP_PATH), on='author_id', how='left')")
    return pmap

# Run after retrieve_stage2():
# persona_map = export_persona_map()


## Preview a Request (no submit)

Build the **first** request of each stage exactly as it will be sent — and print its parts (profile text + attached `gs://` media) — **without** uploading or submitting anything. Use this to sanity-check the multimodal payload (media count, transcript, metadata) before spending a batch job.

In [248]:
def load_approved_taxonomy(path=TAXONOMY_JSON_PATH) -> list:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("status") != "APPROVED":
        raise ValueError(f"Taxonomy at '{path}' is not approved. Review and set status='APPROVED'.")
    taxonomy = data.get("final_taxonomy", [])
    if not taxonomy:
        raise ValueError("final_taxonomy is empty.")
    print(f"✅ Approved taxonomy loaded: {len(taxonomy)} personas.")
    return taxonomy



In [249]:
# === Preview the first request of each stage, WITHOUT submitting ===
def _print_parts(parts):
    print(f"  {len(parts)} part(s):")
    n_media = 0
    for p in parts:
        if "text" in p:
            t = p["text"].strip().replace(chr(10), " ⏎ ")
            print("   text :", (t[:160] + ("…" if len(t) > 160 else "")) if t else "(empty)")
        elif "fileData" in p:
            n_media += 1
            print("   media:", p["fileData"]["fileUri"])
    print(f"  → {n_media} media part(s) attached.")

# ── Stage 1 preview (grouped users + their engaged-post media) ──────────────
print("="*64)
print("STAGE 1 preview — Taxonomy Discovery")
print("="*64)
_s1 = user_features.sample(n=min(SAMPLE_N_USERS, len(user_features)),
                           random_state=SAMPLE_SEED).reset_index(drop=True)
_g0 = _s1.iloc[:STAGE1_USERS_PER_REQUEST]
_line1 = build_stage1_line(_g0)
print(f"request[0] groups {len(_g0)} user(s) | model={MODEL_STAGE1_EXPLORATORY} | "
      f"genCfg={_line1['request']['generationConfig']}")
_print_parts(_line1["request"]["contents"][0]["parts"])
print("  authors in this request:", list(_g0["author_id"].astype(str)))

# ── Stage 2 preview (1 user/request) — needs an APPROVED taxonomy ───────────
print("\n" + "="*64)
print("STAGE 2 preview — Classification")
print("="*64)
try:
    _tax = load_approved_taxonomy(TAXONOMY_JSON_PATH)
    _row = user_features.iloc[0]
    _sys = build_stage2_system_prompt(_tax)
    _line2 = build_stage2_line(_row, _sys)
    print(f"request[0] = author {_row['author_id']} | model={MODEL_STAGE2_CLASSIFY} | "
          f"genCfg={_line2['request']['generationConfig']}")
    _print_parts(_line2["request"]["contents"][0]["parts"])
except FileNotFoundError:
    print("⏭️  No taxonomy.json yet — run Stage 1 + approve before previewing Stage 2.")
except ValueError as _e:
    print(f"⏭️  Stage 2 preview skipped: {_e}")


STAGE 1 preview — Taxonomy Discovery
request[0] groups 3 user(s) | model=gemini-2.5-flash | genCfg={'temperature': 0.0, 'topP': 1.0, 'maxOutputTokens': 8192, 'responseMimeType': 'application/json', 'thinkingConfig': {'thinkingBudget': 1024}}
  5 part(s):
   text : You are an expert community analyst for a major Italian influencer agency. ⏎ Identify distinct audience persona archetypes from Instagram commenter behaviour. ⏎…
   text : USER: 1460351508913112 | Total comments: 1 | Unique posts: 1 | Activity span: 0 days | Avg hrs to comment: 0.0h | Early commenter (<1h): 0% | Reply ratio: 0% | …
   text : USER: 1404418068152050 | Total comments: 1 | Unique posts: 1 | Activity span: 0 days | Avg hrs to comment: 0.0h | Early commenter (<1h): 0% | Reply ratio: 0% | …
   text : USER: 929194886363487 | Total comments: 2 | Unique posts: 2 | Activity span: 17 days | Avg hrs to comment: 0.0h | Early commenter (<1h): 0% | Reply ratio: 0% | …
   text : Identify all distinct behavioural archetypes in

## Pipeline Orchestrator

Single entry point for the Colab Enterprise scheduler. `run_pipeline` submits real batch jobs and
blocks on polling until each completes.

In [259]:
def run_pipeline(mode=PIPELINE_MODE):
    """Submit-only driver. Each stage fires its batch job and returns immediately — then you
    close the laptop and run retrieve_stage1() / retrieve_stage2() later."""
    print(f"\n{'#'*60}\n  SHOW REEL PERSONA PIPELINE (BATCH submit) — MODE: {mode}\n{'#'*60}")
    if mode in ("SAMPLE", "ALL"):
        submit_stage1(user_features)
        if mode == "SAMPLE":
            return None
    if mode in ("FULL", "ALL"):
        taxonomy = load_approved_taxonomy(TAXONOMY_JSON_PATH)
        submit_stage2(user_features, taxonomy)
    return None

result = run_pipeline(PIPELINE_MODE)


############################################################
  SHOW REEL PERSONA PIPELINE (BATCH submit) — MODE: SAMPLE
############################################################

STAGE 1 (batch submit) — Taxonomy Discovery
  Sample: 20000 users | 3/request | Model: gemini-2.5-flash
✅ Stratified Stage-1 sample (user-level dominant sentiment): 20,000 users | strata mix (%):
author_sentiment
positive        79.6
neutral         15.2
negative         5.1
amused           0.1
mixed            0.0
surprise         0.0
question         0.0
joke             0.0
suggestion       0.0
sadness          0.0
affection        0.0
support          0.0
sarcastic        0.0
amusement        0.0
supportive       0.0
anticipation     0.0


Build Stage 1 requests: 100%|██████████| 6667/6667 [00:02<00:00, 2267.89it/s]


  Splitting into 4 batch job(s) to stay under 900,000 tokens each.
[prep] wrote 1,680 requests -> outputs/stage1_input_chunk0.jsonl
[upload] outputs/stage1_input_chunk0.jsonl -> gs://afb_project2/persona_batch/input/stage1_input_chunk0.jsonl
[submit] gemini-2.5-flash -> projects/726524412994/locations/europe-west1/batchPredictionJobs/9171527246106591232  (JOB_STATE_PENDING)
[record] stage1_chunk0 job -> outputs/stage1_chunk0_job.json
[prep] wrote 1,674 requests -> outputs/stage1_input_chunk1.jsonl
[upload] outputs/stage1_input_chunk1.jsonl -> gs://afb_project2/persona_batch/input/stage1_input_chunk1.jsonl
[submit] gemini-2.5-flash -> projects/726524412994/locations/europe-west1/batchPredictionJobs/6622489857014890496  (JOB_STATE_PENDING)
[record] stage1_chunk1 job -> outputs/stage1_chunk1_job.json
[prep] wrote 1,676 requests -> outputs/stage1_input_chunk2.jsonl
[upload] outputs/stage1_input_chunk2.jsonl -> gs://afb_project2/persona_batch/input/stage1_input_chunk2.jsonl
[submit] gemini-

## Retrieve a Finished Batch (run anytime — even after closing & reopening)

Submit is non-blocking, so the Vertex job keeps running on Google's side and writes to
`gs://afb_showreel/persona_batch/output/...` even with your laptop off. Come back later and run the
matching retrieve — it re-fetches the job by the id saved in `outputs/<stage>_job.json`, and only
does work once the job is `SUCCEEDED`:
- **`retrieve_stage1()`** → writes `outputs/taxonomy.json` for review
- **`retrieve_stage2()`** → writes `outputs/user_personas.parquet`

In [263]:
# Status check (no retrieval) — see whether each submitted job is done yet.
import json as _json_sc, os as _os_sc

# Stage 1: may be split into chunks
_manifest_path = f"{LOCAL_DIR}/stage1_chunks.json"
if _os_sc.path.exists(_manifest_path):
    _manifest = _json_sc.load(open(_manifest_path))
    _tags = _manifest["tags"]
elif _os_sc.path.exists(f"{LOCAL_DIR}/stage1_job.json"):
    _tags = ["stage1"]
else:
    _tags = []
    print("stage1: not submitted yet.")

for _tag in _tags:
    try:
        _job = get_recorded_job(_tag)
        print(f"{_tag}: {_job.state}")
    except Exception as _e:
        print(f"{_tag} -> {str(_e)[:120]}")

# Stage 2
if _os_sc.path.exists(f"{LOCAL_DIR}/stage2_job.json"):
    try:
        _job2 = get_recorded_job("stage2")
        print(f"stage2: {_job2.state}")
    except Exception as _e:
        print(f"stage2 -> {str(_e)[:120]}")
else:
    print("stage2: not submitted yet.")


[stage1_chunk0] projects/726524412994/locations/europe-west1/batchPredictionJobs/9171527246106591232 -> JOB_STATE_RUNNING
stage1_chunk0: JOB_STATE_RUNNING
[stage1_chunk1] projects/726524412994/locations/europe-west1/batchPredictionJobs/6622489857014890496 -> JOB_STATE_RUNNING
stage1_chunk1: JOB_STATE_RUNNING
[stage1_chunk2] projects/726524412994/locations/europe-west1/batchPredictionJobs/7748389763857514496 -> JOB_STATE_RUNNING
stage1_chunk2: JOB_STATE_RUNNING
[stage1_chunk3] projects/726524412994/locations/europe-west1/batchPredictionJobs/2240487419583397888 -> JOB_STATE_RUNNING
stage1_chunk3: JOB_STATE_RUNNING
stage2 -> 404 NOT_FOUND. {'error': {'code': 404, 'message': 'The BatchPredictionJob does not exist.', 'status': 'NOT_FOUND'}}


In [56]:
# When the job is SUCCEEDED, run the matching line:
retrieve_stage1()
#retrieve_stage2()

[stage1] projects/840606707685/locations/us-central1/batchPredictionJobs/878465243192229888 -> JOB_STATE_SUCCEEDED
[retrieve] 6,867 response rows from 3 shard(s)
   parse error: Unterminated string starting at: line 6 column 7 (char 248)
   parse error: Expecting value: line 6 column 35 (char 277)
   parse error: Unterminated string starting at: line 7 column 7 (char 249)
   parse error: Unterminated string starting at: line 6 column 7 (char 281)
   parse error: Unterminated string starting at: line 7 column 7 (char 300)
   parse error: Unterminated string starting at: line 7 column 7 (char 260)
   parse error: Expecting value: line 7 column 26 (char 232)
   parse error: Expecting value: line 6 column 32 (char 260)
   parse error: Unterminated string starting at: line 7 column 7 (char 262)
   parse error: Expecting value: line 6 column 33 (char 302)
   parse error: Expecting value: line 6 column 35 (char 258)
   parse error: Expecting value: line 7 column 32 (char 283)
   parse error: 

[{'codename': 'EMOJI_REACTOR',
  'description': 'This user primarily expresses appreciation or amusement through emojis, often with minimal or no text, and may engage with content long after its initial publication.',
  'signals': ['Avg word count: 0-2',
   'Emoji rate: 100%',
   'Reply ratio: 0%',
   'Avg hrs to comment: >300h (can be very late)'],
  'examples': ['👏👏👏😂', '😂']},
 {'codename': 'CASUAL_COMMENTER',
  'description': "This user engages occasionally with short, relevant comments, often including emojis, and sometimes directly addresses the influencer or the post's topic.",
  'signals': ['Total comments: 1-2',
   'Avg word count: 2-5',
   'Emoji rate: 50-100%',
   'Reply ratio: 0%',
   'Activity span: Long (infrequent but sustained)'],
  'examples': ['@camihawke andando dal dentista forse 🤣',
   '@viola_kurmayeva ahahaahahhaahaa']},
 {'codename': 'ENGAGED_REPLIER',
  'description': 'This user actively participates in conversations by replying to other commenters, often sharin

In [67]:
import json
from collections import Counter

# ── Stage 1: discovered candidate codenames (raw + naming-normalised) ──
tax = json.load(open(TAXONOMY_JSON_PATH, encoding="utf-8"))
raw = [c.get("codename","").strip() for c in tax.get("raw_candidates", []) if c.get("codename")]

def norm(c):                      # collapse THE/underscore/case variants
    c = c.upper().replace("_", " ").strip()
    return c[4:].strip() if c.startswith("THE ") else c

raw_cnt  = Counter(raw)
norm_cnt = Counter(norm(c) for c in raw)
print(f"raw unique codenames      : {len(raw_cnt)}  (from {len(raw)} candidates)")
print(f"normalised unique themes  : {len(norm_cnt)}")
print("\nTop themes (normalised) by share:")
tot = sum(norm_cnt.values()) or 1
for name, n in norm_cnt.most_common(20):
    print(f"  {name:<28} {n:>4}  {100*n/tot:5.1f}%")

# ── Stage 2: final persona proportions (after retrieve_stage2) ──
import os
if os.path.exists(RESULTS_PATH):
    df = pd.read_parquet(RESULTS_PATH)
    print(f"\nStage 2 — {df['persona_codename'].nunique()} unique personas over {len(df):,} users:")
    print((df['persona_codename'].value_counts(normalize=True)*100).round(1).astype(str) + ' %')

raw unique codenames      : 4056  (from 19718 candidates)
normalised unique themes  : 2898

Top themes (normalised) by share:
  ADMIRER                       697    3.5%
  ENGAGED FAN                   567    2.9%
  TAGGER                        433    2.2%
  SOCIAL CONNECTOR              418    2.1%
  ENTHUSIAST                    378    1.9%
  CASUAL OBSERVER               331    1.7%
  APPRECIATOR                   328    1.7%
  SUPERFAN                      279    1.4%
  SOCIAL SHARER                 266    1.3%
  EMOJI APPRECIATOR             245    1.2%
  CASUAL APPRECIATOR            243    1.2%
  CHEERLEADER                   235    1.2%
  EMOJI EXPRESSER               222    1.1%
  HUMOR APPRECIATOR             210    1.1%
  LAUGHER                       210    1.1%
  DEVOTED FAN                   189    1.0%
  RELATABLE RESPONDER           176    0.9%
  RELATABLE FAN                 162    0.8%
  CASUAL ADMIRER                161    0.8%
  EMOTIONAL SUPPORTER           160   

## Upload Outputs to GCS

Run after the pipeline completes to persist results across ephemeral runtimes.

In [ ]:
def upload_outputs_to_gcs(local_dir="outputs/"):
    from google.cloud import storage
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    for fname in os.listdir(local_dir):
        local_path = os.path.join(local_dir, fname)
        bucket.blob(fname).upload_from_filename(local_path)
        print(f"   Uploaded: {local_path} -> gs://{GCS_BUCKET}/{fname}")

# upload_outputs_to_gcs()

## Validation & QA

Post-classification health checks: coverage, confidence distribution, and a MECE duplicate-assignment check.

In [ ]:
def validate_classification_output(results_path=RESULTS_PATH):
    df = pd.read_parquet(results_path)
    classified   = df["persona_codename"].notna() & (df["persona_codename"] != "CLASSIFICATION_ERROR")
    coverage_pct = classified.sum() / len(df) * 100

    print("\nCLASSIFICATION QA REPORT")
    print(f"   Total users             : {len(df):,}")
    print(f"   Successfully classified : {classified.sum():,} ({coverage_pct:.1f}%)")
    print(f"   Unclassified / errors   : {(~classified).sum():,}")

    sub = df[classified].copy()
    sub["confidence"] = pd.to_numeric(sub["confidence"], errors="coerce")
    print(f"\n   Confidence  mean: {sub['confidence'].mean():.3f}  "
          f"median: {sub['confidence'].median():.3f}  "
          f"<0.40: {(sub['confidence'] < 0.4).sum():,}")

    dupes = df["author_id"].duplicated().sum()
    print(f"\n   MECE Check: {dupes} duplicate assignments ({'FAIL' if dupes > 0 else 'PASS'})")
    display(df["persona_codename"].value_counts().to_frame("count"))
    return df

# qa_df = validate_classification_output()

## Stage 3 — Macro Persona Discovery (UMAP + HDBSCAN + LLM Naming)

Takes the Stage 2 per-user micro-persona assignments and behavioral features, and discovers
**macro** audience segments through unsupervised ML. Gemini then names each segment as a
human-readable marketing persona.

**Flow:**
1. **Load & merge** — Stage 2 `user_personas.parquet` + Stage 0 `user_features` behavioral matrix.
2. **Feature matrix** — standardised numeric features + one-hot encoded micro-persona label.
3. **UMAP (×2)** — 15-D embedding for HDBSCAN; 2-D embedding for visualisation.
4. **HDBSCAN** — density-based clustering on the 15-D UMAP embedding → macro cluster labels.
5. **Cluster summaries** — dominant micro personas, mean behavioral stats, sample comments.
6. **LLM naming** — single online Gemini call → `codename`, `label`, `description`, `key_traits`, `marketing_insight` per cluster.
7. **Save** — `outputs/user_macro_personas.parquet` (per-user) + `outputs/macro_persona_names.json` (definitions).

> **Tuning knob:** `HDBSCAN_MIN_CLUSTER_SIZE` controls granularity. Re-run HDBSCAN only (no need to redo UMAP) when adjusting.

> **Note (restructured pipeline):** This block is now an *optional* post-Stage-2 macro-clustering **view**. For building the persona **taxonomy**, use **Pathway B** (right after Stage 1) instead — it clusters the stratified sample and writes `final_taxonomy` that Stage 2 consumes.


In [ ]:
!pip install -q umap-learn hdbscan scikit-learn matplotlib

In [ ]:

# ─── Stage 3 configuration ───────────────────────────────────────────────────
# UMAP — two passes: high-D for HDBSCAN, 2-D for scatter plot
UMAP_N_NEIGHBORS          = 30    # larger = more global structure
UMAP_MIN_DIST             = 0.0   # 0.0 packs clusters tighter — ideal for HDBSCAN
UMAP_N_COMPONENTS_CLUSTER = 15    # dims fed into HDBSCAN (15 is a good default)
UMAP_METRIC               = "euclidean"
UMAP_RANDOM_STATE         = 42

# HDBSCAN — tune min_cluster_size to control granularity
# Raise it → fewer, broader macro personas; lower → more granular
HDBSCAN_MIN_CLUSTER_SIZE  = 2000  # ~1% of 194k users; adjust to taste
HDBSCAN_MIN_SAMPLES       = 100
HDBSCAN_CLUSTER_SELECTION = "eom" # excess-of-mass: fewer, more stable clusters
HDBSCAN_CLUSTER_EPSILON   = 0.0   # 0 = default HDBSCAN behaviour

# LLM naming (online call — small payload, no need for batch)
MODEL_STAGE3_NAMING           = "gemini-2.5-flash"
MAX_SAMPLE_USERS_PER_CLUSTER  = 30  # comment samples per cluster fed to LLM

# Outputs
MACRO_PERSONA_PATH = f"{LOCAL_DIR}/user_macro_personas.parquet"
CLUSTER_NAMES_PATH = f"{LOCAL_DIR}/macro_persona_names.json"

print("✅ Stage 3 config loaded.")
print(f"   UMAP   : n_neighbors={UMAP_N_NEIGHBORS}  min_dist={UMAP_MIN_DIST}  "
      f"n_components_cluster={UMAP_N_COMPONENTS_CLUSTER}")
print(f"   HDBSCAN: min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}  "
      f"min_samples={HDBSCAN_MIN_SAMPLES}  method='{HDBSCAN_CLUSTER_SELECTION}'")
print(f"   LLM    : {MODEL_STAGE3_NAMING}  (online call, not batch)")


In [ ]:

import pandas as pd, numpy as np, os

if not os.path.exists(RESULTS_PATH):
    raise FileNotFoundError(
        f"Stage 2 results not found at '{RESULTS_PATH}'. "
        "Run retrieve_stage2() first."
    )
results_df = pd.read_parquet(RESULTS_PATH)
results_df["author_id"] = results_df["author_id"].astype(str)
print(f"Stage 2 results : {len(results_df):,} users  |  "
      f"classified: {results_df['persona_codename'].notna().sum():,}")

# Merge with full behavioral features + comment text from Stage 0 (still in memory)
extra_cols = [
    "author_id",
    "unique_posts_commented", "total_replies_made",
    "pct_comments_under_24h", "emoji_usage_rate", "question_rate",
    "exclamation_rate", "mean_mention_count", "post_concentration_ratio",
    "top_comments_sample",
]
available = [c for c in extra_cols if c in user_features.columns]
feat_ext = user_features[available].copy()
feat_ext["author_id"] = feat_ext["author_id"].astype(str)

stage3_df = results_df.merge(feat_ext, on="author_id", how="left")

# Keep only successfully classified users
stage3_df = stage3_df[
    stage3_df["persona_codename"].notna() &
    (stage3_df["persona_codename"] != "CLASSIFICATION_ERROR")
].copy().reset_index(drop=True)

print(f"Stage 3 working set: {len(stage3_df):,} users with valid micro-persona labels")
print(f"\nMicro-persona distribution:")
print(stage3_df["persona_codename"].value_counts().to_string())


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

# Use the selected features from the feature selection cell above
NUMERIC_FEATURES = SELECTED_NUMERIC_FEATURES.copy()
print(f"Numeric features ({len(NUMERIC_FEATURES)}): {NUMERIC_FEATURES}")

# Standardise (fill any rare NaN with column median first)
num_data = stage3_df[NUMERIC_FEATURES].fillna(stage3_df[NUMERIC_FEATURES].median())
scaler = StandardScaler()
X_num  = scaler.fit_transform(num_data)

# One-hot encode micro persona — weight ×2 so it has comparable influence to numeric block
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_ohe = ohe.fit_transform(stage3_df[["persona_codename"]])
persona_categories = ohe.categories_[0].tolist()
print(f"OHE micro-persona dims: {X_ohe.shape[1]}  →  {persona_categories}")

# One-hot encode dominant_room_vibe if present and non-trivial (>1 unique value)
if "dominant_room_vibe" in stage3_df.columns and stage3_df["dominant_room_vibe"].nunique() > 1:
    ohe_vibe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
    X_vibe = ohe_vibe.fit_transform(stage3_df[["dominant_room_vibe"]])
    vibe_categories = ohe_vibe.categories_[0].tolist()
    print(f"OHE dominant_room_vibe dims: {X_vibe.shape[1]}  →  {vibe_categories}")
    X = np.hstack([X_num, X_ohe * 2.0, X_vibe])
else:
    X_vibe = None
    print("dominant_room_vibe: single value or absent — skipped from OHE")
    X = np.hstack([X_num, X_ohe * 2.0])
print(f"
✅ Feature matrix: {X.shape}  (rows=users, cols=numeric+OHE)")

In [ ]:

import umap

print(f"Running UMAP ({UMAP_N_COMPONENTS_CLUSTER}-D) for HDBSCAN input …")
print(f"  n_neighbors={UMAP_N_NEIGHBORS}  min_dist={UMAP_MIN_DIST}  "
      f"users={len(X):,}  (expect 1-3 min)")

reducer_cluster = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,           # 0.0 packs clusters tighter — better for HDBSCAN
    n_components=UMAP_N_COMPONENTS_CLUSTER,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
    low_memory=True,
    verbose=True,
)
X_umap_cluster = reducer_cluster.fit_transform(X)
print(f"\n✅ UMAP clustering embedding: {X_umap_cluster.shape}")

print(f"\nRunning UMAP (2-D) for visualisation …")
reducer_viz = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=0.1,
    n_components=2,
    metric=UMAP_METRIC,
    random_state=UMAP_RANDOM_STATE,
    low_memory=True,
    verbose=False,
)
X_umap_viz = reducer_viz.fit_transform(X)
print(f"✅ UMAP 2-D embedding: {X_umap_viz.shape}")

stage3_df["umap_x"] = X_umap_viz[:, 0]
stage3_df["umap_y"] = X_umap_viz[:, 1]


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.cm as cm

personas  = stage3_df["persona_codename"].unique()
palette   = cm.get_cmap("tab20", len(personas))
color_map = {p: palette(i) for i, p in enumerate(sorted(personas))}

fig, ax = plt.subplots(figsize=(12, 9))
for p, grp in stage3_df.groupby("persona_codename"):
    ax.scatter(grp["umap_x"], grp["umap_y"],
               s=2, alpha=0.4, color=color_map[p], label=p)

ax.set_title("UMAP 2-D projection — coloured by micro persona (Stage 2 labels)", fontsize=13)
ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
ax.legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc="upper left",
          fontsize=8, framealpha=0.7)
plt.tight_layout()
plt.savefig(f"{LOCAL_DIR}/umap_micro_personas.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {LOCAL_DIR}/umap_micro_personas.png")


In [ ]:

import hdbscan

print(f"Running HDBSCAN on {UMAP_N_COMPONENTS_CLUSTER}-D UMAP embedding …")
print(f"  min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}  "
      f"min_samples={HDBSCAN_MIN_SAMPLES}  "
      f"cluster_selection_method='{HDBSCAN_CLUSTER_SELECTION}'")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    cluster_selection_method=HDBSCAN_CLUSTER_SELECTION,
    cluster_selection_epsilon=HDBSCAN_CLUSTER_EPSILON,
    prediction_data=True,
)
labels = clusterer.fit_predict(X_umap_cluster)

stage3_df["macro_cluster"] = labels
unique_clusters = sorted(set(labels[labels >= 0]))
n_noise = int((labels == -1).sum())

print(f"\n✅ HDBSCAN complete.")
print(f"   Clusters found  : {len(unique_clusters)}")
print(f"   Clustered users : {(labels >= 0).sum():,}  ({100*(labels >= 0).mean():.1f}%)")
print(f"   Noise (−1)      : {n_noise:,}  ({100*n_noise/len(labels):.1f}%)")
print("\n   Cluster sizes:")
for c in unique_clusters:
    cnt = int((labels == c).sum())
    print(f"     Cluster {c:>3}: {cnt:>7,} users  ({100*cnt/len(labels):.1f}%)")

# Tip: if you get too many / too few clusters, adjust HDBSCAN_MIN_CLUSTER_SIZE in config
# and re-run this cell. No need to rerun UMAP.


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ── Left: macro clusters ──────────────────────────────────────────────────
palette_macro = cm.get_cmap("Set1", max(len(unique_clusters), 1))
noise_color   = (0.75, 0.75, 0.75, 0.25)

noise_mask = stage3_df["macro_cluster"] == -1
if noise_mask.any():
    axes[0].scatter(stage3_df.loc[noise_mask, "umap_x"],
                    stage3_df.loc[noise_mask, "umap_y"],
                    s=1, alpha=0.15, color=noise_color, label="noise", zorder=1)

for i, c in enumerate(unique_clusters):
    mask = stage3_df["macro_cluster"] == c
    axes[0].scatter(stage3_df.loc[mask, "umap_x"],
                    stage3_df.loc[mask, "umap_y"],
                    s=2, alpha=0.5, color=palette_macro(i),
                    label=f"Cluster {c}  (n={mask.sum():,})", zorder=2)

axes[0].set_title("Macro clusters — HDBSCAN", fontsize=12)
axes[0].set_xlabel("UMAP-1"); axes[0].set_ylabel("UMAP-2")
axes[0].legend(markerscale=6, fontsize=8)

# ── Right: micro personas ─────────────────────────────────────────────────
for p, grp in stage3_df.groupby("persona_codename"):
    axes[1].scatter(grp["umap_x"], grp["umap_y"],
                    s=2, alpha=0.4, color=color_map[p], label=p)

axes[1].set_title("Micro personas — LLM (Stage 2)", fontsize=12)
axes[1].set_xlabel("UMAP-1"); axes[1].set_ylabel("UMAP-2")
axes[1].legend(markerscale=5, fontsize=7,
               bbox_to_anchor=(1.01, 1), loc="upper left")

plt.suptitle("UMAP projection: macro clusters vs. micro personas", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f"{LOCAL_DIR}/umap_macro_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved -> {LOCAL_DIR}/umap_macro_clusters.png")


In [ ]:

def build_cluster_summary(cluster_id: int, df: pd.DataFrame) -> dict:
    sub = df[df["macro_cluster"] == cluster_id].copy()
    n   = len(sub)

    # Micro-persona composition
    persona_dist = (
        sub["persona_codename"].value_counts(normalize=True)
                               .mul(100).round(1)
                               .head(5).to_dict()
    )

    # Mean behavioral stats
    stat_cols = [c for c in [
        "total_comments", "activity_span_days", "mean_hours_to_comment",
        "pct_comments_under_1h", "reply_ratio", "mean_word_count",
        "emoji_usage_rate", "question_rate", "exclamation_rate",
    ] if c in sub.columns]
    mean_stats = sub[stat_cols].mean().round(3).to_dict()

    # Representative comment samples from top_comments_sample
    samples = []
    if "top_comments_sample" in sub.columns:
        sample_rows = sub["top_comments_sample"].dropna().sample(
            n=min(MAX_SAMPLE_USERS_PER_CLUSTER, n), random_state=42
        )
        for txt in sample_rows:
            frags = [f.strip() for f in str(txt).split("|||") if f.strip()]
            samples.extend(frags[:2])
            if len(samples) >= 30:
                break
    samples = samples[:30]

    # Stage 2 justification snippets
    justifications = []
    if "justification" in sub.columns:
        for j in sub["justification"].dropna().sample(n=min(10, n), random_state=42):
            justifications.append(str(j)[:200])

    return {
        "cluster_id":              cluster_id,
        "n_users":                 n,
        "pct_audience":            round(100 * n / max(len(df[df["macro_cluster"] >= 0]), 1), 1),
        "dominant_micro_personas": persona_dist,
        "mean_behavioral_stats":   mean_stats,
        "sample_comments":         samples,
        "sample_justifications":   justifications[:10],
    }


cluster_summaries = [build_cluster_summary(c, stage3_df) for c in unique_clusters]
print(f"✅ Cluster summaries built for {len(cluster_summaries)} clusters:\n")
for s in cluster_summaries:
    top_p = list(s["dominant_micro_personas"].items())[:3]
    print(f"  Cluster {s['cluster_id']:>3}  n={s['n_users']:>7,}  ({s['pct_audience']:.1f}%)"
          f"  top micro-personas: {top_p}")


In [ ]:

from google.genai import types as genai_types

STAGE3_NAMING_SYSTEM = (
    "You are a strategic audience analyst for an Italian influencer marketing agency.\n"
    "You have clustered Instagram commenters into macro audience segments using UMAP + HDBSCAN.\n"
    "Each cluster is described by its dominant micro-persona mix, mean behavioral stats, "
    "and representative comment samples.\n\n"
    "For EACH cluster produce ONE macro persona with exactly these fields:\n"
    "  cluster_id        : (integer — echo back exactly as provided)\n"
    "  codename          : UPPER_SNAKE_CASE concise name (e.g. PASSIONATE_LOYALISTS)\n"
    "  label             : Short Title Case marketing label (3-5 words)\n"
    "  description       : 2-3 sentences — who they are, how and why they engage\n"
    "  key_traits        : JSON list of 4-6 behavioural / attitudinal bullet strings\n"
    "  marketing_insight : 1 actionable sentence for the brand or agency\n\n"
    "Output ONLY a valid JSON array of objects with exactly those 6 keys. "
    "No preamble, no markdown fences, no extra keys."
)

def build_naming_prompt(summaries: list) -> str:
    blocks = []
    for s in summaries:
        stats_str = "  ".join(
            f"{k}={v:.2f}" if isinstance(v, float) else f"{k}={v}"
            for k, v in list(s["mean_behavioral_stats"].items())[:7]
        )
        persona_str = "  ".join(f"{p}: {pct}%" for p, pct in s["dominant_micro_personas"].items())
        comments_str = "\n    • ".join(s["sample_comments"][:8])
        justs = s.get("sample_justifications", [])
        just_str = ("\n  LLM justification samples: " +
                    " | ".join(justs[:3])) if justs else ""
        blocks.append(
            f"CLUSTER {s['cluster_id']}  (n={s['n_users']:,}, {s['pct_audience']}% of clustered audience)\n"
            f"  Dominant micro-personas: {persona_str}\n"
            f"  Mean behavioral stats:   {stats_str}{just_str}\n"
            f"  Sample comments:\n    • {comments_str}"
        )
    return STAGE3_NAMING_SYSTEM + "\n\n=== CLUSTER DATA ===\n\n" + "\n\n".join(blocks)

prompt = build_naming_prompt(cluster_summaries)

cfg = genai_types.GenerateContentConfig(
    temperature=0.0,
    top_p=1.0,
    max_output_tokens=4096,
    response_mime_type="application/json",
    thinking_config=genai_types.ThinkingConfig(thinking_budget=2048),
)

print(f"Calling {MODEL_STAGE3_NAMING} to name {len(cluster_summaries)} macro clusters …")
resp = client.models.generate_content(model=MODEL_STAGE3_NAMING, contents=prompt, config=cfg)

raw = strip_fences(resp.text or "")
try:
    macro_personas = json.loads(raw)
except json.JSONDecodeError as e:
    print("⚠️  JSON parse error:", e)
    print("Raw response (first 800 chars):\n", raw[:800])
    macro_personas = []

if not isinstance(macro_personas, list):
    print("⚠️  Expected JSON array, got:", type(macro_personas).__name__)
    macro_personas = []

print(f"\n✅ {len(macro_personas)} macro personas named:\n")
for mp in macro_personas:
    print(f"  Cluster {mp.get('cluster_id', '?'):>3}  "
          f"{str(mp.get('codename', '')):.<40}  {mp.get('label', '')}")
    print(f"           {mp.get('description', '')[:130]}")
    print(f"           Insight: {mp.get('marketing_insight', '')[:130]}\n")

with open(CLUSTER_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "status":        "COMPLETE",
        "n_clusters":    len(unique_clusters),
        "noise_users":   int(n_noise),
        "macro_personas": macro_personas,
    }, f, ensure_ascii=False, indent=2)
print(f"✅ Macro persona definitions saved -> {CLUSTER_NAMES_PATH}")


In [ ]:

# Build cluster_id -> macro persona lookup
id_to_macro = {}
for mp in macro_personas:
    cid = mp.get("cluster_id")
    if cid is not None:
        id_to_macro[int(cid)] = {
            "macro_persona_codename": mp.get("codename", f"CLUSTER_{cid}"),
            "macro_persona_label":    mp.get("label", ""),
        }

stage3_df["macro_persona_codename"] = stage3_df["macro_cluster"].map(
    lambda c: id_to_macro.get(c, {}).get("macro_persona_codename", "NOISE")
)
stage3_df["macro_persona_label"] = stage3_df["macro_cluster"].map(
    lambda c: id_to_macro.get(c, {}).get("macro_persona_label", "Noise / Unclustered")
)

# Save per-user macro assignments
output_cols = [
    "author_id", "persona_codename", "confidence",
    "macro_cluster", "macro_persona_codename", "macro_persona_label",
    "umap_x", "umap_y",
]
output_cols = [c for c in output_cols if c in stage3_df.columns]
stage3_df[output_cols].to_parquet(MACRO_PERSONA_PATH, index=False)

print(f"✅ Macro persona assignments saved -> {MACRO_PERSONA_PATH}")
print(f"\nMacro Persona Distribution (clustered users only):")
clustered = stage3_df[stage3_df["macro_cluster"] >= 0]
dist = (clustered["macro_persona_codename"].value_counts(normalize=True).mul(100).round(1))
for codename, pct in dist.items():
    cid = int(clustered.loc[clustered["macro_persona_codename"] == codename, "macro_cluster"].iloc[0])
    label = id_to_macro.get(cid, {}).get("macro_persona_label", "")
    print(f"  {str(codename):<42} {pct:>6.1f}%  —  {label}")
print(f"\n  Noise / unclustered: {(stage3_df['macro_cluster'] == -1).sum():,} users")
